# Great Lakes Basin MODFLOW 6 — Clean, ordered workflow

This notebook is a **reorganized version** of the input-preparation and model-run workflow.
It keeps the core logic from the original notebook, but it puts sections in a cleaner order,
removes most ad-hoc debugging cells, and adds clearer section comments so it is easier to
rerun from top to bottom.

## Recommended execution order
Run the notebook **sequentially** from top to bottom.  
If you change `id2d` / `idomain`, rerun all downstream sections that depend on the active domain:
- recharge
- GHB
- DRN
- plotting
- model build/run


## 0) Imports


In [ ]:
import os
import gc
import time
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"  # helps reading NetCDF/HDF5 from network drives on Windows
# remove conflicting PROJ environment variables
os.environ.pop("PROJ_LIB", None)
os.environ.pop("PROJ_DATA", None)

print("PROJ_LIB =", os.environ.get("PROJ_LIB"))
print("PROJ_DATA =", os.environ.get("PROJ_DATA"))
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning, module="pyogrio")
warnings.filterwarnings("ignore", category=DeprecationWarning)
from pathlib import Path
import re, glob, shutil, tempfile, calendar

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio as rio
from rasterio.transform import from_origin, from_bounds
from rasterio.features import rasterize
from rasterio.warp import reproject, Resampling
import shapely.geos
from shapely.geometry import MultiLineString
import matplotlib.pyplot as plt
import matplotlib as mpl
from rasterio.crs import CRS
import flopy
from flopy.utils.gridintersect import GridIntersect
from affine import Affine
import xarray as xr

print(f"numpy version: {np.__version__}")
print(f"matplotlib version: {mpl.__version__}")
print(f"flopy version: {flopy.__version__}")

## 1) User settings

This section holds file paths, model options, package settings, and run controls.
Edit only this section when you need to change inputs or outputs.


In [ ]:
# --- Model identity ---
nameSim   = "Greatlakes"
nameModel = "Testing_6"  # MF6 model name; package files will be Testing.dis, Testing.rch, etc.

# --- MF6 executable ---
bindir = Path(r"D:\Users\abolmaal\modelling\Modflow\helper")
exe_path = str((bindir / "mf6.exe").resolve())

# --- Simulation workspace (will be RECREATED) ---
dirModelFilesBase = r"D:\Users\abolmaal\modelling\Modflow"
sim_ws = str(Path(dirModelFilesBase) / nameModel)

# --- Boundary polygon (truth) ---
boundary_shp = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Ibound\extended_Bdry_final_GLB_Albers_exported.shp"

IBOUND = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Ibound\Idomain_mask_30m.tif"
# --- Raw rasters (any CRS/resolution; we will warp to template) ---
nameInputTop       = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\DEM\DEM_extended20kmbdr_1000m.tif"

nameInputLayBot    = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Bottom\modelbottom.tif"

nameInputHorizCond = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\HK\HK_5band_1000m.tif"


# this is the actual starting head raster
nameInputStrt     = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Wells\starting_heads_clamped_1000m.tif"

#this is your lake/land mask, NOT starting heads
#nameInputMask   = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Costantheads\domain_water_mask_30m_buff2000m.tif"

# --- CHD / DRN vector inputs ---
pathInputConstHead      = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Costantheads\CHD_cells_points_dem.shp"
fieldInputConstHeadElev = "head"

# -- GHB ---
OUT_GHB_TABLE = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Lakes\GreatLakes_GHB_cells.csv"
OUT_STAGE_TABLE = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Lakes\GreatLakes_stage_monthly_for_model.csv"

LAKES_SHP = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Lakes\GreatLakes.shp"


# pathInputDrn = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\NHD\streams_3174_clip_to_modelgrid.shp"
# fieldInputDrnWidth = "WIDTH_M"
USE_DRN = True
USE_WETLAND_DRN = False   # separate wetland DRN no longer needed

nameInputDrainElev = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Drains\drain_elevation.tif"
drain_elev_aligned = os.path.join(sim_ws, "drain_elevation_aligned.tif")
drain_presence_raw = os.path.join(sim_ws, "drain_presence_raw.tif")
drain_frac_aligned = os.path.join(sim_ws, "drain_fraction_aligned.tif")


DRN_K_DIVISOR = 1.0       # use 1.0 if you want exactly Kcell
# DRN_K_DIVISOR = 10.0    # use this instead if you want Kcell/10 for vertical leakage

DRN_MIN_THICK = 0.1       # minimum cell thickness used in conductance
DRN_MIN_AREA_FRAC = 0.005  # skip tiny drain fractions, use a value below the stream fraction to avoid float precision filtering
DRN_COND_MULT = 1.0
DRN_ELEV_EPS = 0.01       # keep drain elevation slightly inside the cell

# --- NLDAS monthly Noah (NetCDF) ---
nldas_root = Path(r"D:\Users\abolmaal\Data\Downloaded\Climatedata\Gridded\NLDAS_NOAHVIC_M.2.0")
NLDAS_VAR  = "Qsb"

# --- Model grid definition ---
CELL = 1000       # meters (try 2000 or 1000 later)
EPSG = 3174       # NAD_1983_Great_Lakes_Basin_Albers

# --- Outputs (template + warped rasters) ---
GRID_DIR    = Path(r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\GRID_3174")
ALIGNED_DIR = Path(r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\ALIGNED_3174")
GRID_DIR.mkdir(parents=True, exist_ok=True)
ALIGNED_DIR.mkdir(parents=True, exist_ok=True)

template_tif = str(GRID_DIR / f"template_{CELL}m_epsg{EPSG}.tif")
idomain_tif  = str(GRID_DIR / f"idomain_{CELL}m_epsg{EPSG}.tif")

top_aligned   = str(ALIGNED_DIR / f"TOP_{CELL}m.tif")
botm_aligned  = str(ALIGNED_DIR / f"BOTM_{CELL}m.tif")
hk_aligned    = str(ALIGNED_DIR / f"HK_{CELL}m.tif")
mask_aligned  = str(ALIGNED_DIR / f"MASK_{CELL}m.tif")
strt_aligned  = str(ALIGNED_DIR / f"STRT_{CELL}m.tif")
Ibound_aligned = str(ALIGNED_DIR / f"IBOUND_{CELL}m.tif")

# --- Streams source for DRN build ---
gdb_path = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\NHD\streams_tmp.gdb"
layer_name = "streams_3174"   # change if your FileGDB layer name differs

# --- Time controls ---
START_DATE = "2020-01-01"
END_DATE   = "2023-12-01"

NPER_TEST =13          # set None for full run
USE_FIVE_LAYER_MODEL = True
FORCE_CONSTANT_CHD = False
LAKE_STAGE = 100.0
USE_GHB = True
USE_DRN = True

# =========================================================
# WETLAND DRN SETTINGS
# =========================================================
# USE_WETLAND_DRN = True

# WETLANDS_SHP = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Wetlands\Wetlands_GL_NoLakes.shp"

# WETLAND_TSED_M = 1.0
# WETLAND_KV_DIVISOR = 10.0
# WETLAND_DEPTH_BELOW_LAND_M = 0.1

# Figure_dir 
fig_dir = r"D:\Users\abolmaal\modelling\Figs\testing6"

## 2) Helper functions

All reusable helper functions are grouped here so the rest of the notebook can stay short and readable.
This includes:
- grid/template helpers
- raster read/write helpers
- recharge helpers
- GHB helpers
- DRN helper builders


In [ ]:

def snap_bounds_to_cell(bounds, cell):
    xmin, ymin, xmax, ymax = bounds
    xmin = np.floor(xmin / cell) * cell
    ymin = np.floor(ymin / cell) * cell
    xmax = np.ceil(xmax / cell) * cell
    ymax = np.ceil(ymax / cell) * cell
    return float(xmin), float(ymin), float(xmax), float(ymax)

def build_model_months(start_date, nper):
    return pd.date_range(start=start_date, periods=nper, freq="MS")

def build_monthly_perioddata(start="2000-01-01", end="2025-12-01", nstp=1, tsmult=1.0):
    months = pd.date_range(start=start, end=end, freq="MS")
    pddata = []
    for d in months:
        ndays = calendar.monthrange(d.year, d.month)[1]
        pddata.append((float(ndays), int(nstp), float(tsmult)))
    return pddata, months

def make_template_from_boundary(boundary_shp, out_template_tif, cellsize):
    gdf = gpd.read_file(boundary_shp)

    if gdf.crs is None:
        raise ValueError(f"Boundary file has no CRS: {boundary_shp}")

    xmin, ymin, xmax, ymax = snap_bounds_to_cell(gdf.total_bounds, cellsize)

    width = int(round((xmax - xmin) / cellsize))
    height = int(round((ymax - ymin) / cellsize))

    from rasterio.transform import from_origin
    transform = from_origin(xmin, ymax, cellsize, cellsize)

    profile = {
        "driver": "GTiff",
        "height": height,
        "width": width,
        "count": 1,
        "dtype": "int16",
        "transform": transform,
        "nodata": 0,
        "compress": "deflate",
        "tiled": True,
        "BIGTIFF": "YES",
    }

    if os.path.exists(out_template_tif):
        try:
            os.remove(out_template_tif)
        except PermissionError:
            print("Could not delete locked file:", out_template_tif)

    with rio.open(out_template_tif, "w", **profile) as dst:
        import numpy as np
        dst.write(np.zeros((height, width), dtype=np.int16), 1)

    return out_template_tif


def rasterize_idomain(boundary_shp, template_tif, out_idomain_tif, burn_value=1, all_touched=False):
    """
    Rasterize boundary directly onto template grid.
    Assumes boundary_shp is already in the same CRS as the intended grid.
    """
    gdf = gpd.read_file(boundary_shp)

    if gdf.crs is None:
        raise ValueError(f"Boundary file has no CRS: {boundary_shp}")

    with rio.open(template_tif) as tmp:
        arr = rasterize(
            shapes=[(geom, burn_value) for geom in gdf.geometry if geom is not None and not geom.is_empty],
            out_shape=(tmp.height, tmp.width),
            transform=tmp.transform,
            fill=0,
            dtype="int32",
            all_touched=all_touched
        )

        profile = tmp.profile.copy()
        profile.update(
            driver="GTiff",
            dtype="int32",
            count=1,
            nodata=0,
            compress="deflate",
            tiled=True,
            BIGTIFF="YES"
        )

        # optional: strip fields that sometimes cause trouble when copied through
        profile.pop("blockxsize", None)
        profile.pop("blockysize", None)

    if os.path.exists(out_idomain_tif):
        try:
            os.remove(out_idomain_tif)
        except PermissionError:
            print("Could not delete locked file:", out_idomain_tif)

    with rio.open(out_idomain_tif, "w", **profile) as dst:
        dst.write(arr, 1)

    return out_idomain_tif

def assert_match_template_no_crs(raster_tif, template_tif, name="Raster"):
    """
    Check shape and transform only. Skip CRS because Rasterio cannot
    reliably read CRS in the current environment.
    """
    with rio.open(raster_tif) as a, rio.open(template_tif) as b:
        if (a.height, a.width) != (b.height, b.width):
            raise AssertionError(
                f"{name} shape mismatch: {(a.height, a.width)} vs {(b.height, b.width)}"
            )
        if a.transform != b.transform:
            raise AssertionError(
                f"{name} transform mismatch:\n{a.transform}\nvs\n{b.transform}"
            )
    print(f"{name} matches template in shape and transform.")




def warp_raster_to_template(src_path, template_path, out_path, resampling, dst_nodata=-9999.0):
    with rio.open(template_path) as tmpl, rio.open(src_path) as src:
        dst_meta = src.meta.copy()
        dst_meta.update(
            driver="GTiff",
            crs=tmpl.crs,
            transform=tmpl.transform,
            width=tmpl.width,
            height=tmpl.height,
            nodata=dst_nodata,
            compress="deflate",
            tiled=True,
            BIGTIFF="YES",
            blockxsize=256,
            blockysize=256,
        )
        with rio.open(out_path, "w", **dst_meta) as dst:
            for b in range(1, src.count + 1):
                reproject(
                    source=rio.band(src, b),
                    destination=rio.band(dst, b),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    src_nodata=src.nodata,
                    dst_transform=tmpl.transform,
                    dst_crs=tmpl.crs,
                    dst_nodata=dst_nodata,
                    resampling=resampling,
                )
    return out_path

def assert_match_template(path, template_path, name):
    with rio.open(template_path) as t, rio.open(path) as s:
        assert (s.height, s.width) == (t.height, t.width), f"{name} shape mismatch"
        assert s.crs == t.crs, f"{name} CRS mismatch"
        assert s.transform == t.transform, f"{name} transform mismatch"

def read_band1(path, dtype="float32"):
    with rio.open(path) as src:
        return src.read(1).astype(dtype, copy=False), src.nodata

def read_all_bands(path, dtype="float32"):
    with rio.open(path) as src:
        return src.read().astype(dtype, copy=False), src.nodata

def clean_continuous(a, nodata, fill=0.0):
    a = a.astype("float32", copy=False)
    a = np.nan_to_num(a, nan=fill, posinf=fill, neginf=fill)
    if nodata is not None:
        a = np.where(a == nodata, fill, a)
    return a

def make_gridintersect(modelgrid):
    try:
        return GridIntersect(modelgrid, method="vertex")
    except TypeError:
        try:
            return GridIntersect(modelgrid, "vertex")
        except TypeError:
            return GridIntersect(modelgrid)

def update_ghb_k_from_idomain(ghb_cells_df, idomain):
    """
    Update GHB layer k so each boundary cell is assigned to the first active layer
    at its (i, j) location based on the final DIS idomain.
    """
    df = ghb_cells_df.copy()

    if {"i", "j"}.issubset(df.columns):
        i_col, j_col = "i", "j"
    elif {"row", "col"}.issubset(df.columns):
        i_col, j_col = "row", "col"
        if "i" not in df.columns:
            df["i"] = df["row"]
        if "j" not in df.columns:
            df["j"] = df["col"]
    else:
        raise ValueError("ghb_cells_df must contain either (i, j) or (row, col) columns.")

    if idomain.ndim == 2:
        active = idomain > 0
        keep_idx = []
        new_k = []
        for idx, r in enumerate(df.itertuples(index=False)):
            i = int(getattr(r, i_col))
            j = int(getattr(r, j_col))
            if 0 <= i < active.shape[0] and 0 <= j < active.shape[1] and active[i, j]:
                keep_idx.append(idx)
                new_k.append(0)
        df = df.iloc[keep_idx].copy()
        df["k"] = new_k
        return df.reset_index(drop=True)

    if idomain.ndim == 3:
        nlay, nrow, ncol = idomain.shape
        keep_idx = []
        new_k = []
        for idx, r in enumerate(df.itertuples(index=False)):
            i = int(getattr(r, i_col))
            j = int(getattr(r, j_col))
            if not (0 <= i < nrow and 0 <= j < ncol):
                continue
            active_layers = np.where(idomain[:, i, j] > 0)[0]
            if len(active_layers) == 0:
                continue
            keep_idx.append(idx)
            new_k.append(int(active_layers[0]))
        df = df.iloc[keep_idx].copy()
        df["k"] = new_k
        return df.reset_index(drop=True)

    raise ValueError(f"idomain must be 2D or 3D, got shape {idomain.shape}")

def assign_ghb_k_from_stage_floor(ghb_cells_df, idomain, botm3d, stage_floor_by_name, stage_margin=0.05):
    """
    Choose the shallowest active layer whose bottom is below the minimum
    stage for that lake/stage_name.

    Parameters
    ----------
    ghb_cells_df : DataFrame
        Must contain stage_name, i, j
    idomain : ndarray
        3D array (nlay, nrow, ncol)
    botm3d : ndarray
        3D bottom array (nlay, nrow, ncol)
    stage_floor_by_name : pandas Series or dict
        Minimum stage for each stage_name across the modeled period
    stage_margin : float
        Small safety margin so stage > bottom + margin
    """
    if idomain.ndim != 3:
        raise ValueError("assign_ghb_k_from_stage_floor expects 3D idomain")

    df = ghb_cells_df.copy()
    keep_rows = []
    new_k = []
    dropped = []

    nlay, nrow, ncol = idomain.shape

    for idx, r in enumerate(df.itertuples(index=False)):
        i = int(r.i)
        j = int(r.j)
        sname = str(r.stage_name)

        if sname not in stage_floor_by_name:
            dropped.append((idx, "missing_stage_name"))
            continue

        if not (0 <= i < nrow and 0 <= j < ncol):
            dropped.append((idx, "out_of_bounds"))
            continue

        stage_floor = float(stage_floor_by_name[sname])

        active_layers = np.where(idomain[:, i, j] > 0)[0]
        if len(active_layers) == 0:
            dropped.append((idx, "no_active_layers"))
            continue

        valid_layers = [
            int(k) for k in active_layers
            if float(botm3d[k, i, j]) < (stage_floor - stage_margin)
        ]

        if len(valid_layers) == 0:
            dropped.append((idx, "no_layer_below_stage"))
            continue

        keep_rows.append(idx)
        new_k.append(valid_layers[0])   # shallowest valid layer

    out = df.iloc[keep_rows].copy().reset_index(drop=True)
    out["k"] = new_k

    print("GHB cells kept after stage-aware layer assignment:", len(out))
    print("GHB cells dropped:", len(dropped))
    if len(dropped) > 0:
        print(pd.Series([d[1] for d in dropped]).value_counts())

    return out


# ---- NLDAS indexing ----
def index_blend_qsb_monthlies(root: Path):
    files = sorted(root.rglob("BLEND_Qsb_A*.nc"))
    rows = []
    for f in files:
        m = re.search(r"_A(\d{6})\.nc$", f.name)
        if not m:
            continue
        yyyymm = m.group(1)
        dt = pd.Timestamp(int(yyyymm[:4]), int(yyyymm[4:6]), 1)
        rows.append((dt, str(f)))
    df = pd.DataFrame(rows, columns=["date", "path"]).sort_values("date").reset_index(drop=True)
    return df

def copy_to_local_cached(src_path: str) -> str:
    src = Path(src_path)
    cache_dir = Path(tempfile.gettempdir()) / "nldas_nc_cache"
    cache_dir.mkdir(parents=True, exist_ok=True)
    dst = cache_dir / src.name
    if (not dst.exists()) or (dst.stat().st_size != src.stat().st_size) or (dst.stat().st_mtime < src.stat().st_mtime):
        shutil.copy2(src, dst)
    return str(dst)

def read_qsb_lat_lon_attrs(nc_path: str, var="Qsb"):
    local_path = copy_to_local_cached(nc_path)

    last_err = None
    for engine in ("h5netcdf", "netcdf4"):
        try:
            ds = xr.open_dataset(local_path, engine=engine, decode_times=False)
            da = ds[var]
            if "time" in da.dims:
                da = da.isel(time=0)

            units = (da.attrs.get("units") or "").strip()
            cell_methods = (da.attrs.get("cell_methods") or "").strip()

            qsb = da.values.astype("float32")
            lat = da["lat"].values
            lon = da["lon"].values
            ds.close()
            return qsb, lat, lon, units, cell_methods
        except Exception as e:
            last_err = e

    raise RuntimeError(f"Failed to read {nc_path}. Last error: {last_err}")

def parse_yyyymm_from_filename(fname: str):
    base = Path(fname).name
    m = re.search(r"\.A(\d{6})\.", base)
    if not m:
        m = re.search(r"_A(\d{6})\.nc$", base)
    if not m:
        raise ValueError(f"Cannot parse YYYYMM from {fname}")
    yyyymm = m.group(1)
    return int(yyyymm[:4]), int(yyyymm[4:6])

def qsb_month_to_rech_mday_on_template(
    nc_path,
    template_tif,
    var,
    id2d,
    dst_crs_wkt,
    clamp_negative_to_zero=True,
    src_crs="+proj=longlat +datum=WGS84 +no_defs +type=crs",
    resampling=Resampling.bilinear,
):
    ds = xr.open_dataset(nc_path)

    if var not in ds:
        raise KeyError(f"Variable '{var}' not found in {nc_path}")

    da = ds[var].squeeze()
    vals = np.array(da.values, dtype="float32")

    lat_name = None
    lon_name = None
    for nm in da.dims:
        low = nm.lower()
        if "lat" in low or low == "y":
            lat_name = nm
        if "lon" in low or low == "x":
            lon_name = nm

    if lat_name is None or lon_name is None:
        raise ValueError(f"Could not identify lat/lon dims in {da.dims}")

    lats = np.array(ds[lat_name].values, dtype=float)
    lons = np.array(ds[lon_name].values, dtype=float)

    # north-up raster convention
    if lats[0] < lats[-1]:
        vals = np.flipud(vals)
        lats = lats[::-1]

    dx = float(np.abs(lons[1] - lons[0]))
    dy = float(np.abs(lats[1] - lats[0]))
    x0 = float(lons.min() - dx / 2.0)
    y0 = float(lats.max() + dy / 2.0)

    src_transform = Affine(dx, 0.0, x0, 0.0, -dy, y0)

    # ---------------------------------------------------------
    # CORRECT UNIT CONVERSION
    # Qsb units = kg m-2 with cell_methods = time: sum
    # -> equivalent to mm/month
    # -> convert to m/day
    # ---------------------------------------------------------
    time_val = None
    if "time" in ds.coords and ds["time"].size > 0:
        time_val = pd.to_datetime(ds["time"].values[0])
    else:
        # fallback: infer from filename if needed
        time_val = pd.Timestamp(nc_path.split(".A")[1][:6] + "01")

    days_in_month = int(time_val.days_in_month)

    rech_src = np.array(vals, dtype="float32") / 1000.0 / days_in_month
    rech_src = np.nan_to_num(rech_src, nan=0.0, posinf=0.0, neginf=0.0)

    if clamp_negative_to_zero:
        rech_src[rech_src < 0] = 0.0

    with rio.open(template_tif) as tmp:
        dst = np.zeros((tmp.height, tmp.width), dtype="float32")

        reproject(
            source=rech_src,
            destination=dst,
            src_transform=src_transform,
            src_crs=src_crs,
            dst_transform=tmp.transform,
            dst_crs=dst_crs_wkt,
            resampling=resampling,
            dst_nodata=0.0,
        )

    dst = np.nan_to_num(dst, nan=0.0, posinf=0.0, neginf=0.0).astype("float32")
    dst[dst < 0] = 0.0
    dst[id2d <= 0] = 0.0

    return dst

def build_rch_spd_from_index(df_nc, months_run, template_tif, id2d, var, dst_crs_wkt):
    rch_spd = {}

    for per, dt in enumerate(months_run):
        hit = df_nc.loc[df_nc["date"] == dt]

        if hit.empty:
            with rio.open(template_tif) as tmp:
                rch_spd[per] = np.zeros((tmp.height, tmp.width), dtype="float32")
            continue

        nc_path = hit.iloc[0]["path"]

        rch_spd[per] = qsb_month_to_rech_mday_on_template(
            nc_path=nc_path,
            template_tif=template_tif,
            var=var,
            id2d=id2d,
            dst_crs_wkt=dst_crs_wkt,
        )

    return rch_spd

def intersect_grid_feature(ix, pathFeature, lay=0, addFields=None, grid_crs=None):
    addFields = addFields or []
    try:
        gdf = gpd.read_file(pathFeature, engine="fiona")
    except Exception:
        gdf = gpd.read_file(pathFeature)
        
    if gdf.empty:
        return pd.DataFrame()

    if grid_crs is not None and gdf.crs is not None and gdf.crs != grid_crs:
        gdf = gdf.to_crs(grid_crs)

    try:
        gdf = gdf.explode(index_parts=False).reset_index(drop=True)
    except TypeError:
        gdf = gdf.explode().reset_index(drop=True)

    gdf = gdf[gdf.geometry.notna() & (~gdf.geometry.is_empty)].copy()

    try:
        gdf["geometry"] = gdf.geometry.buffer(0)
    except Exception:
        pass

    parts = []
    for i in range(len(gdf)):
        geom = gdf.geometry.iloc[i]
        if geom is None or geom.is_empty:
            continue
        try:
            df = pd.DataFrame(ix.intersect(geom, geo_dataframe=False))
        except TypeError:
            df = pd.DataFrame(ix.intersect(geom))
        if df.empty:
            continue

        for f in addFields:
            if f in gdf.columns:
                df[f] = gdf[f].iloc[i]

        df["cellids"] = df["cellids"].apply(lambda rc: (lay, rc[0], rc[1]))
        df["row"] = df["cellids"].apply(lambda x: x[1])
        df["col"] = df["cellids"].apply(lambda x: x[2])
        parts.append(df)

    if not parts:
        return pd.DataFrame()
    return pd.concat(parts, ignore_index=True)

def compute_thickness(top2d, botm3d):
    thick = np.zeros_like(botm3d, dtype=float)
    thick[0] = top2d - botm3d[0]
    for k in range(1, botm3d.shape[0]):
        thick[k] = botm3d[k - 1] - botm3d[k]
    return thick

def ensure_3d(arr, nlay, nrow, ncol):
    arr = np.asarray(arr)
    if arr.ndim == 0:
        return np.full((nlay, nrow, ncol), float(arr))
    elif arr.ndim == 2:
        return np.repeat(arr[np.newaxis, :, :], nlay, axis=0)
    elif arr.ndim == 3:
        return arr
    else:
        raise ValueError(f"Unexpected array ndim: {arr.ndim}")

def get_extent(xorigin, yorigin, delr, delc, nrow, ncol):
    xmin = xorigin
    xmax = xorigin + np.sum(delr)
    ymin = yorigin
    ymax = yorigin + np.sum(delc)
    return [xmin, xmax, ymin, ymax]

def mask_model_array(arr2d, idomain_layer):
    out = np.array(arr2d, dtype=float)
    out[idomain_layer <= 0] = np.nan
    return out

def get_date_labels(start_date, nper):
    return pd.date_range(start=start_date, periods=nper, freq="MS")

def get_water_table(head_t, idomain, huge=1e20):
    nlay, nrow, ncol = head_t.shape
    wt = np.full((nrow, ncol), np.nan, dtype=float)
    for k in range(nlay):
        hk = np.array(head_t[k], dtype=float)
        hk[(idomain[k] <= 0) | (hk > huge)] = np.nan
        take = np.isnan(wt) & np.isfinite(hk)
        wt[take] = hk[take]
    return wt

def plot_bc_masks(chd_rec, drn_rec, xorigin, yorigin, delr, delc, nrow, ncol, idomain=None):
    extent = get_extent(xorigin, yorigin, delr, delc, nrow, ncol)

    if idomain is not None:
        bg = np.where(idomain[0] > 0, 1.0, np.nan)
    else:
        bg = np.ones((nrow, ncol), dtype=float)

    chd_mask = np.full((nrow, ncol), np.nan, dtype=float)
    drn_mask = np.full((nrow, ncol), np.nan, dtype=float)

    for (k, r, c), _ in chd_rec:
        if int(k) == 0:
            chd_mask[int(r), int(c)] = 1.0

    for rec in drn_rec:
        (k, r, c), elev, cond = rec
        if int(k) == 0:
            drn_mask[int(r), int(c)] = 1.0

    plt.figure(figsize=(10, 8))
    plt.imshow(bg, origin="upper", extent=extent, alpha=0.15, cmap="Greys")
    plt.imshow(chd_mask, origin="upper", extent=extent, alpha=0.9, cmap="Blues")
    plt.imshow(drn_mask, origin="upper", extent=extent, alpha=0.8, cmap="Reds")
    plt.title("Boundary-condition cells (CHD=blue, DRN=red)")
    plt.xlabel("Easting (m)")
    plt.ylabel("Northing (m)")
    plt.tight_layout()
    plt.show()

def safe_rmtree(folder, tries=10, wait=1.0):
    folder = Path(folder)
    if not folder.exists():
        return
    gc.collect()
    time.sleep(0.2)
    last_err = None
    for _ in range(tries):
        try:
            shutil.rmtree(folder)
            return
        except Exception as e:
            last_err = e
            gc.collect()
            time.sleep(wait)
    raise last_err

def extract_kij(rec_list):
    """
    Extract (k, i, j) from MODFLOW-style boundary record lists.

    Supports records like:
      ((k, i, j), head, cond, ...)
      ((k, i, j), elev, cond, ...)
    """
    if rec_list is None or len(rec_list) == 0:
        return np.empty((0, 3), dtype=int)

    out = []
    for rec in rec_list:
        try:
            cellid = rec[0]
            k, i, j = cellid
            out.append((int(k), int(i), int(j)))
        except Exception:
            # skip malformed records
            continue

    if len(out) == 0:
        return np.empty((0, 3), dtype=int)

    return np.array(out, dtype=int)


def build_lake_mask(LAKES_SHP, grid_crs, xorigin, yorigin, delr, delc, nrow, ncol):
    lakes = gpd.read_file(LAKES_SHP)

    if lakes.crs is not None:
        try:
            lakes = lakes.to_crs(grid_crs)
        except Exception:
            lakes = lakes.to_crs(str(grid_crs))

    ymax = yorigin + np.sum(delc)
    transform = from_origin(xorigin, ymax, float(delr[0]), float(delc[0]))

    lake_mask = rasterize(
        [(geom, 1) for geom in lakes.geometry if geom is not None and not geom.is_empty],
        out_shape=(nrow, ncol),
        transform=transform,
        fill=0,
        default_value=1,
        dtype="uint8",
    ).astype(bool)

    return lake_mask

def save_or_show(fig_dir, filename):
    if save_figs:
        outpath = os.path.join(fig_dir, filename)
        fig.savefig(outpath, dpi=300, bbox_inches="tight")
        print("Saved figure:", outpath)
    plt.show()
    
    
    
# function for wetlands 
def build_wetland_drn(
    pathWetlands,
    ix,
    grid_crs,
    idomain,
    top2d,
    botm3d,
    hk3d,
    delr,
    delc,
    WETLAND_TSED_M=1.0,
    WETLAND_KV_DIVISOR=10.0,
    WETLAND_DEPTH_BELOW_LAND_M=0.1,
    MIN_KV=1e-8,
):
    """
    Build wetland drain cells from wetland polygons intersected with model grid.

    Conductance:
        C = K * Af / Tsed
    where
        K   = proxy vertical K from top layer = hk3d[0] / WETLAND_KV_DIVISOR
        Af  = wetland overlap area in the cell
        Tsed= assumed wetland-bottom sediment thickness

    Drain elevation:
        top2d - WETLAND_DEPTH_BELOW_LAND_M,
        but never below cell bottom + 0.1 m
    """
    import numpy as np
    import pandas as pd

    wet = intersect_grid_feature(
        ix=ix,
        pathFeature=pathWetlands,
        lay=0,
        addFields=[],
        grid_crs=grid_crs,
    )

    if wet.empty:
        raise ValueError("Wetland-grid intersection returned no cells.")

    # overlap area
    if "areas" in wet.columns:
        wet["Af"] = pd.to_numeric(wet["areas"], errors="coerce")
    elif "area" in wet.columns:
        wet["Af"] = pd.to_numeric(wet["area"], errors="coerce")
    else:
        raise ValueError("Could not find overlap area column ('areas' or 'area').")

    wet = wet[wet["Af"].notna() & (wet["Af"] > 0)].copy()

    # current grid indices
    wet["i"] = wet["row"].astype(int)
    wet["j"] = wet["col"].astype(int)
    wet["k"] = 0

    # keep active top-layer cells only
    rr = wet["i"].to_numpy(dtype=int)
    cc = wet["j"].to_numpy(dtype=int)
    keep = (
        (rr >= 0) & (rr < idomain.shape[1]) &
        (cc >= 0) & (cc < idomain.shape[2]) &
        (idomain[0, rr, cc] > 0)
    )
    wet = wet.loc[keep].copy()

    if wet.empty:
        raise ValueError("All wetland intersections were removed by top-layer idomain.")

    rr = wet["i"].to_numpy(dtype=int)
    cc = wet["j"].to_numpy(dtype=int)

    # K proxy
    kv = hk3d[0, rr, cc].astype(float) / float(WETLAND_KV_DIVISOR)
    kv = np.maximum(kv, MIN_KV)

    # drain elevation
    elev = top2d[rr, cc].astype(float) - float(WETLAND_DEPTH_BELOW_LAND_M)
    elev = np.maximum(elev, botm3d[0, rr, cc].astype(float) + 0.1)

    # conductance
    cond = kv * wet["Af"].to_numpy(dtype=float) / float(WETLAND_TSED_M)

    wet["elev"] = elev
    wet["kv"] = kv
    wet["cond"] = cond

    # collapse duplicates by cell
    dfWetDrn = (
        wet.groupby(["k", "i", "j"], as_index=False)
        .agg(
            Af=("Af", "sum"),
            elev=("elev", "min"),
            kv=("kv", "first"),
            cond=("cond", "sum"),
        )
    )

    dfWetDrn = dfWetDrn[dfWetDrn["cond"] > 0].copy()

    wet_drn_rec = [
        ((int(r.k), int(r.i), int(r.j)), float(r.elev), float(r.cond))
        for r in dfWetDrn.itertuples(index=False)
    ]

    return dfWetDrn, wet_drn_rec





In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio as rio
import fiona

def build_cellid_set_from_df(df):
    if df is None or len(df) == 0:
        return set()
    d = df.copy()
    if "cellids" not in d.columns:
        d["cellids"] = list(zip(d["lay"].astype(int), d["row"].astype(int), d["col"].astype(int)))
    return set(d["cellids"])

def build_cellid_set_from_rec(rec):
    out = set()
    if rec is None:
        return out
    for r in rec:
        cellid = r[0]
        if isinstance(cellid, tuple) and len(cellid) == 3:
            out.add((int(cellid[0]), int(cellid[1]), int(cellid[2])))
    return out

def get_template_info(template_tif):
    with rio.open(template_tif) as src:
        b = src.bounds
        transform = src.transform
        dx = abs(transform.a)
        dy = abs(transform.e)
    return (b.left, b.bottom, b.right, b.top), dx, dy

def build_drn_from_streams_glb(
    ix,
    gdb_path,
    layer_name,
    template_tif,
    id2d,
    top2d,
    botm3d,
    hk3d=None,
    lay=0,

    width_field="WIDTH_M",
    fcode_field="FCODE",
    permanency_field="PERMANENCY",
    flowclass_field="FLOW_CLASS",

    default_width=10.0,

    min_width=None,               # e.g. 25.0 or 10.0
    keep_permanency=None,         # e.g. ["Perennial"] after checking actual values
    keep_flowclass=None,          # e.g. ["StreamRiver"] after checking actual values
    keep_fcodes=None,
    drop_fcodes=None,

    simplify_tol=100.0,
    min_len=None,
    min_len_frac=0.30,

    elev_offset=0.5,
    bed_thick=1.0,
    bed_k=0.1,
    cond_mult=1.0,
    cond_cap=5e4,

    dfChd=None,
    chd_rec=None,
    report_every=1000,
):
    bbox_tuple, dx, dy = get_template_info(template_tif)
    cellsize = float(min(dx, dy))

    if min_len is None:
        min_len = float(min_len_frac) * cellsize

    print("Template cell size:", cellsize)
    print("Using min_len:", min_len)

    print("Available layers in GDB (first 30):")
    print(fiona.listlayers(gdb_path)[:30])

    gdf = gpd.read_file(
        gdb_path,
        layer=layer_name,
        bbox=bbox_tuple,
        engine="fiona",
    )

    if gdf.empty:
        raise RuntimeError("No stream features read in model bbox.")

    print("Loaded features:", len(gdf))
    print("Columns:", gdf.columns.tolist())

    gdf = gdf[gdf.geometry.notna() & (~gdf.geometry.is_empty)].copy()
    gdf = gdf[gdf.geometry.geom_type.isin(["LineString", "MultiLineString"])].copy()

    if gdf.empty:
        raise RuntimeError("No valid LineString/MultiLineString features remain.")

    if simplify_tol is not None and simplify_tol > 0:
        gdf["geometry"] = gdf.geometry.simplify(simplify_tol, preserve_topology=False)
        gdf = gdf[gdf.geometry.notna() & (~gdf.geometry.is_empty)].copy()

    # width
    if width_field in gdf.columns:
        gdf["width_m"] = pd.to_numeric(gdf[width_field], errors="coerce")
    else:
        gdf["width_m"] = np.nan

    gdf["width_m"] = gdf["width_m"].fillna(default_width)
    gdf.loc[~np.isfinite(gdf["width_m"]), "width_m"] = float(default_width)
    gdf.loc[gdf["width_m"] <= 0, "width_m"] = float(default_width)

    print("Width unique:", np.unique(gdf["width_m"])[:20])

    if min_width is not None:
        before = len(gdf)
        gdf = gdf[gdf["width_m"] >= float(min_width)].copy()
        print(f"Filter width >= {min_width}: {before} -> {len(gdf)}")

    # permanency
    if keep_permanency is not None and permanency_field in gdf.columns:
        before = len(gdf)
        gdf = gdf[gdf[permanency_field].isin(keep_permanency)].copy()
        print(f"Filter PERMANENCY in {keep_permanency}: {before} -> {len(gdf)}")

    # flow class
    if keep_flowclass is not None and flowclass_field in gdf.columns:
        before = len(gdf)
        gdf = gdf[gdf[flowclass_field].isin(keep_flowclass)].copy()
        print(f"Filter FLOW_CLASS in {keep_flowclass}: {before} -> {len(gdf)}")

    # fcode
    if fcode_field in gdf.columns:
        if keep_fcodes is not None:
            before = len(gdf)
            gdf = gdf[gdf[fcode_field].isin(list(keep_fcodes))].copy()
            print(f"Keep FCodes {keep_fcodes}: {before} -> {len(gdf)}")

        if drop_fcodes is not None:
            before = len(gdf)
            gdf = gdf[~gdf[fcode_field].isin(list(drop_fcodes))].copy()
            print(f"Drop FCodes {drop_fcodes}: {before} -> {len(gdf)}")

    if gdf.empty:
        raise RuntimeError("All stream features were removed by filtering.")

    # intersect feature-by-feature
    acc = {}
    geoms = gdf.geometry.values
    widths = gdf["width_m"].to_numpy(dtype=float)

    print("Intersecting filtered stream features with grid...")
    for n, (geom, width_m) in enumerate(zip(geoms, widths), start=1):
        if geom is None or geom.is_empty:
            continue

        try:
            out = ix.intersect(geom, geo_dataframe=False)
        except TypeError:
            out = ix.intersect(geom)

        df = pd.DataFrame(out)
        if df.empty:
            continue

        if "lengths" in df.columns:
            length_col = "lengths"
        elif "length" in df.columns:
            length_col = "length"
        else:
            raise RuntimeError(f"No line length column returned. Got: {df.columns.tolist()}")

        if "cellids" in df.columns:
            cellid_col = "cellids"
        elif "cellid" in df.columns:
            cellid_col = "cellid"
        else:
            raise RuntimeError(f"No cellid column returned. Got: {df.columns.tolist()}")

        lens = pd.to_numeric(df[length_col], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        cellids = df[cellid_col].tolist()

        for cellid, L in zip(cellids, lens):
            if L <= 0:
                continue

            if isinstance(cellid, tuple):
                if len(cellid) == 2:
                    row, col = int(cellid[0]), int(cellid[1])
                elif len(cellid) == 3:
                    row, col = int(cellid[-2]), int(cellid[-1])
                else:
                    continue
            else:
                continue

            key = (int(lay), row, col)
            if key not in acc:
                acc[key] = {"lengths": 0.0, "wlen": 0.0}

            acc[key]["lengths"] += float(L)
            acc[key]["wlen"] += float(width_m) * float(L)

        if n % report_every == 0:
            print(f"  intersected {n}/{len(gdf)} features...")

    if len(acc) == 0:
        raise RuntimeError("No DRN intersections found after filtering.")

    rows_out = []
    for (lay_i, row_i, col_i), vals in acc.items():
        total_len = vals["lengths"]
        avg_width = vals["wlen"] / total_len if total_len > 0 else default_width
        rows_out.append((lay_i, row_i, col_i, total_len, avg_width))

    dfDrn = pd.DataFrame(rows_out, columns=["lay", "row", "col", "lengths", "width_m"])

    before = len(dfDrn)
    dfDrn = dfDrn[dfDrn["lengths"] >= float(min_len)].copy()
    print(f"Drop cells with stream length < {min_len} m: {before} -> {len(dfDrn)}")

    if dfDrn.empty:
        raise RuntimeError("All DRN cells removed by min_len.")

    rr = dfDrn["row"].to_numpy(dtype=int)
    cc = dfDrn["col"].to_numpy(dtype=int)

    before = len(dfDrn)
    dfDrn = dfDrn[np.asarray(id2d)[rr, cc] == 1].copy()
    print(f"Keep only active cells: {before} -> {len(dfDrn)}")

    if dfDrn.empty:
        raise RuntimeError("No active DRN cells remain.")

    chd_set = set()
    chd_set |= build_cellid_set_from_df(dfChd)
    chd_set |= build_cellid_set_from_rec(chd_rec)

    dfDrn["cellids"] = list(zip(dfDrn["lay"].astype(int), dfDrn["row"].astype(int), dfDrn["col"].astype(int)))

    if len(chd_set) > 0:
        before = len(dfDrn)
        dfDrn = dfDrn[~dfDrn["cellids"].isin(chd_set)].copy()
        print(f"Remove DRN cells overlapping CHD: {before} -> {len(dfDrn)}")

    rr = dfDrn["row"].to_numpy(dtype=int)
    cc = dfDrn["col"].to_numpy(dtype=int)

    TOP = np.asarray(top2d, dtype=float)[rr, cc]
    BOT = np.asarray(botm3d, dtype=float)[lay, rr, cc]
    W = dfDrn["width_m"].to_numpy(dtype=float)
    L = dfDrn["lengths"].to_numpy(dtype=float)

    elev = np.maximum(BOT + 0.1, TOP - float(elev_offset))
    Kbed = np.full(len(dfDrn), float(bed_k), dtype=float)

    cond = Kbed * (W * L) / float(bed_thick)
    cond = cond * float(cond_mult)

    if cond_cap is not None:
        cond = np.minimum(cond, float(cond_cap))

    dfDrn["elev"] = np.nan_to_num(elev, nan=0.0, posinf=0.0, neginf=0.0)
    dfDrn["cond"] = np.nan_to_num(cond, nan=0.0, posinf=0.0, neginf=0.0)

    dfDrn = dfDrn[dfDrn["cond"] > 0].copy()

    drn_rec = [
        ((int(r.lay), int(r.row), int(r.col)), float(r.elev), float(r.cond))
        for r in dfDrn.itertuples(index=False)
    ]

    print("Final DRN records:", len(drn_rec))
    print("cond min/max:", float(dfDrn["cond"].min()), float(dfDrn["cond"].max()))

    return dfDrn, drn_rec


def build_drn_from_raster(
    drain_elev2d,
    drain_frac2d,
    idomain,
    top2d,
    botm3d,
    hk3d,
    delr,
    delc,
    k_divisor=1.0,
    cond_mult=1.0,
    min_thick=0.1,
    min_area_frac=0.01,
    elev_eps=0.01,
):
    recs = []
    rows = []

    nlay, nrow, ncol = idomain.shape

    ii, jj = np.where(np.isfinite(drain_elev2d) & (drain_frac2d >= min_area_frac))

    for i, j in zip(ii, jj):
        active_k = np.where(idomain[:, i, j] > 0)[0]
        if len(active_k) == 0:
            continue
        k = int(active_k[0])

        cell_top = float(top2d[i, j] if k == 0 else botm3d[k - 1, i, j])
        cell_bot = float(botm3d[k, i, j])
        cell_thick = max(cell_top - cell_bot, min_thick)

        elev = float(drain_elev2d[i, j])

        # keep drain elevation inside the cell
        if elev >= cell_top:
            elev = cell_top - elev_eps
        if elev <= cell_bot:
            continue
        # calculate the conductance based on the fraction of the cell area covered by the drain and the cell's hydraulic conductivity 
        area = float(delr[j] * delc[i] * drain_frac2d[i, j])
        if area <= 0:
            continue

        kcell = float(hk3d[k, i, j]) / k_divisor
        cond = float(kcell * area / cell_thick * cond_mult)

        if not np.isfinite(cond) or cond <= 0:
            continue

        rec = ((k, int(i), int(j)), elev, cond)
        recs.append(rec)

        rows.append({
            "lay": k,
            "row": int(i),
            "col": int(j),
            "elev": elev,
            "cond": cond,
            "area_m2": area,
            "area_frac": float(drain_frac2d[i, j]),
            "kcell": kcell,
            "cell_thick": cell_thick,
        })

    df = pd.DataFrame(rows)
    return df, recs

## 3) Build template grid and initial IDOMAIN

This creates the model template raster from the boundary polygon and rasterizes the initial active domain.
At this stage, the domain is only the first-pass rasterized basin footprint.


In [ ]:
make_template_from_boundary(boundary_shp, template_tif, CELL)
rasterize_idomain(boundary_shp, template_tif, idomain_tif, burn_value=1, all_touched=False)
assert_match_template_no_crs(idomain_tif, template_tif, "IDOMAIN")

In [ ]:
id2d, _ = read_band1(idomain_tif, dtype="int32")
id2d = (id2d > 0).astype(np.int32)
print("Active cells:", int(np.count_nonzero(id2d)))

with rio.open(template_tif) as tmp:
    print("Template shape:", (tmp.height, tmp.width), "CRS:", tmp.crs, "transform:", tmp.transform)
    nrow, ncol = tmp.height, tmp.width
    delr = np.full(ncol, tmp.transform.a, dtype=float)
    delc = np.full(nrow, abs(tmp.transform.e), dtype=float)
    xorigin = tmp.bounds.left
    yorigin = tmp.bounds.bottom
    grid_crs = tmp.crs

## 4) Build binary drain-presence raster from the drain-elevation raster

This converts the raw drain-elevation raster into a 0/1 presence raster.
Important: only **nonzero, finite** drain-elevation cells are treated as drain presence.


In [ ]:


# drain_presence_raw = os.path.join(sim_ws, "drain_presence_raw.tif")

# with rio.open(nameInputDrainElev) as src:
#     nd = src.nodata
#     meta = src.meta.copy()

#     meta.update(
#         driver="GTiff",
#         dtype="uint8",
#         count=1,
#         nodata=0,
#         compress="lzw"
#     )

#     meta.pop("blockxsize", None)
#     meta.pop("blockysize", None)

#     with rio.open(drain_presence_raw, "w", **meta) as dst:
#         for _, window in src.block_windows(1):
#             arr = src.read(1, window=window)

#             if nd is not None:
#                 pres = (
#                     np.isfinite(arr) &
#                     (arr != nd) &
#                     (arr != 0)
#                 ).astype("uint8")
#             else:
#                 pres = (
#                     np.isfinite(arr) &
#                     (arr != 0)
#                 ).astype("uint8")

#             dst.write(pres, 1, window=window)

# print("Created drain presence raster:", drain_presence_raw)

In [ ]:
drain_presence_raw = os.path.join(sim_ws, "drain_presence_raw.tif")

# Assumed stream width for cells where drain elevation exists
# At 1000 m cell size, a typical stream width of 30-100 m
# gives a fraction of 0.03 to 0.10
STREAM_WIDTH_M    = 8.0    # assumed average stream/river width in metres
CELL_SIZE_M       = float(CELL)   # 1000 m

# fraction = stream_width / cell_size
# (assumes stream runs the full length of the cell)
STREAM_FRAC = STREAM_WIDTH_M / CELL_SIZE_M   # = 0.01 for 10m stream in 1000m cell

print(f"Cell size           : {CELL_SIZE_M:.0f} m")
print(f"Assumed stream width: {STREAM_WIDTH_M:.0f} m")
print(f"Drain fraction used : {STREAM_FRAC:.4f}  ({STREAM_FRAC*100:.1f}% of cell)")

with rio.open(nameInputDrainElev) as src:
    nd = src.nodata
    meta = src.meta.copy()

    meta.update(
        driver="GTiff",
        dtype="float32",
        count=1,
        nodata=0.0,
        compress="lzw"
    )
    meta.pop("blockxsize", None)
    meta.pop("blockysize", None)

    with rio.open(drain_presence_raw, "w", **meta) as dst:
        for _, window in src.block_windows(1):
            arr = src.read(1, window=window).astype(float)

            if nd is not None:
                valid = (
                    np.isfinite(arr) &
                    (arr != nd) &
                    (arr != 0) &
                    (arr > -9000)
                )
            else:
                valid = (
                    np.isfinite(arr) &
                    (arr != 0) &
                    (arr > -9000)
                )

            # assign fraction where drain exists, 0 elsewhere
            frac = np.where(valid, STREAM_FRAC, 0.0).astype("float32")
            dst.write(frac, 1, window=window)

print(f"Created drain fraction raster : {drain_presence_raw}")
print(f"Valid drain cells in source   : 589,365 → fraction = {STREAM_FRAC:.4f}")

## 5) Warp all continuous rasters to the common template grid

This aligns all major inputs to the same cell size, extent, and transform as the model template.


In [ ]:
# Continuous rasters
warp_raster_to_template(nameInputTop,       template_tif, top_aligned,    Resampling.bilinear, dst_nodata=-9999.0)
warp_raster_to_template(nameInputLayBot,    template_tif, botm_aligned,   Resampling.bilinear, dst_nodata=-9999.0)
warp_raster_to_template(nameInputHorizCond, template_tif, hk_aligned,     Resampling.nearest,  dst_nodata=-9999.0)

# Skip old IBOUND completely
# Ibound_aligned = IBOUND

# Starting heads
warp_raster_to_template(nameInputStrt, template_tif, strt_aligned, Resampling.bilinear, dst_nodata=-9999.0)

# Drains
warp_raster_to_template(nameInputDrainElev, template_tif, drain_elev_aligned, Resampling.nearest, dst_nodata=-9999.0)

warp_raster_to_template(
    drain_presence_raw,
    template_tif,
    drain_frac_aligned,
    Resampling.average,
    dst_nodata=0.0
)

# Assertions
assert_match_template(top_aligned,         template_tif, "TOP")
assert_match_template(botm_aligned,        template_tif, "BOTM")
assert_match_template(hk_aligned,          template_tif, "HK")
assert_match_template(strt_aligned,        template_tif, "STRT")
assert_match_template(idomain_tif,         template_tif, "IDOMAIN")
assert_match_template(drain_elev_aligned,  template_tif, "DRAIN_ELEV")
assert_match_template(drain_frac_aligned,  template_tif, "DRAIN_FRAC")

print("All required rasters aligned to template ✅")

## 6) Read aligned rasters, sanitize nodata values, and build the core model arrays

This section:
- reads TOP / BOTM / HK / STRT / DRN rasters
- removes bad sentinel values like `-9999`
- builds `id2d` and `idomain`
- maps HK bands to model layers
- deactivates cells with invalid thickness


In [ ]:
top2d, top_nd     = read_band1(top_aligned, dtype="float32")
botm_raw, botm_nd = read_all_bands(botm_aligned, dtype="float32")
hk_raw, hk_nd     = read_all_bands(hk_aligned, dtype="float32")

# starting-head raster: interpolated heads
strt2d_raw, strt_nd = read_band1(strt_aligned, dtype="float32")

# ---------------------------------------------------------
# clean continuous rasters
# ---------------------------------------------------------
top2d  = clean_continuous(top2d,    top_nd,  fill=np.nan)
botm3d = clean_continuous(botm_raw, botm_nd, fill=np.nan)
hk_raw = clean_continuous(hk_raw,   hk_nd,   fill=0.0)

# explicit cleanup for bad sentinel values that may survive metadata cleaning
top2d = np.array(top2d, dtype=float)
top2d[~np.isfinite(top2d)] = np.nan
top2d[top2d <= -9000] = np.nan

botm3d = np.array(botm3d, dtype=float)
botm3d[~np.isfinite(botm3d)] = np.nan
botm3d[botm3d <= -9000] = np.nan

hk_raw = np.array(hk_raw, dtype=float)
hk_raw[~np.isfinite(hk_raw)] = 0.0
hk_raw[hk_raw <= -9000] = 0.0

# No lake/land classification in this workflow
# Keep placeholders so later code that expects these variables will still work
mask2d = np.zeros_like(id2d, dtype="float32")
water_mask = np.zeros_like(id2d, dtype=bool)

# keep missing starting heads as NaN so you can fill them later
strt2d_raw = clean_continuous(strt2d_raw, strt_nd, fill=np.nan)
strt2d_raw = np.array(strt2d_raw, dtype=float)
strt2d_raw[~np.isfinite(strt2d_raw)] = np.nan
strt2d_raw[strt2d_raw <= -9000] = np.nan

nlay = botm3d.shape[0]
print("nlay from BOTM bands:", nlay, "| HK bands:", hk_raw.shape[0])

# ---------------------------------------------------------
# read drain rasters
# ---------------------------------------------------------
drain_elev2d, drain_nd = read_band1(drain_elev_aligned, dtype="float32")
drain_frac2d, frac_nd  = read_band1(drain_frac_aligned, dtype="float32")

drain_elev2d = np.array(drain_elev2d, dtype=float)
drain_elev2d[~np.isfinite(drain_elev2d)] = np.nan
if drain_nd is not None:
    drain_elev2d[drain_elev2d == drain_nd] = np.nan
drain_elev2d[drain_elev2d <= -9000] = np.nan

drain_frac2d = np.array(drain_frac2d, dtype=float)
drain_frac2d[~np.isfinite(drain_frac2d)] = 0.0
if frac_nd is not None:
    drain_frac2d[drain_frac2d == frac_nd] = 0.0
drain_frac2d[drain_frac2d <= -9000] = 0.0
drain_frac2d = np.clip(drain_frac2d, 0.0, 1.0)

# keep drains only inside active model cells
drain_frac2d[id2d <= 0] = 0.0
drain_elev2d[id2d <= 0] = np.nan

# ---------------------------------------------------------
# Map HK to layers
# ---------------------------------------------------------
if hk_raw.shape[0] == nlay:
    hk3d = hk_raw.copy()
elif nlay == 1 and hk_raw.shape[0] >= 1:
    hk3d = hk_raw[0:1, :, :].copy()
elif hk_raw.shape[0] == 1 and nlay > 1:
    hk3d = np.repeat(hk_raw, nlay, axis=0)
else:
    hk3d = np.zeros((nlay, nrow, ncol), dtype="float32")
    for k in range(nlay):
        hk3d[k] = hk_raw[min(k, hk_raw.shape[0] - 1)]

# ---------------------------------------------------------
# Inactive -> 0
# ---------------------------------------------------------
top2d[id2d == 0] = 0.0
botm3d[:, id2d == 0] = np.nan
hk3d[:, id2d == 0] = 0.0

# ---------------------------------------------------------
# clamp HK in active
# ---------------------------------------------------------
hk_min, hk_max = 1e-6, 1e4
m = (id2d == 1)
for k in range(nlay):
    hk3d[k, m] = np.clip(hk3d[k, m], hk_min, hk_max)
    hk3d[k, ~m] = 0.0

# ---------------------------------------------------------
# Thickness checks -> deactivate bad cells
# ---------------------------------------------------------
bad = np.zeros_like(id2d, dtype=bool)

thk1 = top2d - botm3d[0]
bad |= (id2d == 1) & (~np.isfinite(thk1) | (thk1 <= 0))

for k in range(1, nlay):
    thk = botm3d[k - 1] - botm3d[k]
    bad |= (id2d == 1) & (~np.isfinite(thk) | (thk <= 0))

print("Bad thickness cells:", int(bad.sum()))

id2d = id2d.copy()
id2d[bad] = 0
idomain = np.repeat(id2d[np.newaxis, :, :], nlay, axis=0).astype(np.int32)
print("Active after thickness:", int(np.count_nonzero(id2d)))

# zero-out inactive arrays after bad-cell removal
top2d[id2d == 0] = 0.0
hk3d[:, id2d == 0] = 0.0
drain_frac2d[id2d <= 0] = 0.0
drain_elev2d[id2d <= 0] = np.nan

# ---------------------------------------------------------
# Starting heads: if start looks like a mask, use top; else use start
# ---------------------------------------------------------
active_start = strt2d_raw[id2d == 1]
if active_start.size == 0 or np.all(~np.isfinite(active_start)):
    use_start = False
else:
    uvals = np.unique(active_start[np.isfinite(active_start)])[:10]
    use_start = not (
        np.all(np.isin(uvals, [0.0, 1.0])) or
        (np.nanmax(active_start) <= 2.0)
    )

# fill missing starting-head cells with top as fallback
strt2d = np.where(np.isfinite(strt2d_raw), strt2d_raw, top2d)

# expand to 3D
strt = np.repeat(strt2d[np.newaxis, :, :], nlay, axis=0).astype("float32")
strt[idomain == 0] = 0.0

print("Using START raster for heads?", use_start)

# quick diagnostics for sentinel cleanup
print("Bottom min after cleanup:", np.nanmin(botm3d))
print("Any bottom <= -9000 left?", bool(np.any(botm3d <= -9000)))

if USE_FIVE_LAYER_MODEL:
    print("\nSwitching to 5-layer structure to match 5 HK bands ...")
    botm_base = botm3d[0].copy()
    thk_total = top2d - botm_base

    fractions = np.array([0.15, 0.20, 0.25, 0.20, 0.20], dtype="float32")
    fractions = fractions / fractions.sum()
    min_thk = 5.0

    bad_total = (id2d == 1) & (~np.isfinite(thk_total) | (thk_total <= (min_thk * 5)))
    print("Cells with insufficient total thickness:", int(bad_total.sum()))
    id2d = id2d.copy()
    id2d[bad_total] = 0

    thk_total = top2d - botm_base
    thk_layers = fractions[:, None, None] * thk_total[None, :, :]
    thk_layers = np.maximum(thk_layers, min_thk)

    cum = np.cumsum(thk_layers, axis=0)
    botm3d = top2d[None, :, :] - cum
    botm3d[-1, :, :] = botm_base

    nlay = hk_raw.shape[0]
    hk3d = hk_raw.copy()
    idomain = np.repeat(id2d[np.newaxis, :, :], nlay, axis=0).astype(np.int32)

    top2d[id2d == 0] = 0.0
    botm3d[:, id2d == 0] = np.nan
    hk3d[:, id2d == 0] = 0.0

    hk_min, hk_max = 1e-8, 1e4
    m = (id2d == 1)
    for k in range(nlay):
        hk3d[k, m] = np.clip(hk3d[k, m], hk_min, hk_max)
        hk3d[k, ~m] = 0.0

    bad = np.zeros_like(id2d, dtype=bool)
    thk1 = top2d - botm3d[0]
    bad |= (id2d == 1) & (~np.isfinite(thk1) | (thk1 <= 0))

    for k in range(1, nlay):
        thk = botm3d[k - 1] - botm3d[k]
        bad |= (id2d == 1) & (~np.isfinite(thk) | (thk <= 0))

    print("Bad thickness cells (after building 5 layers):", int(bad.sum()))
    id2d[bad] = 0
    idomain = np.repeat(id2d[np.newaxis, :, :], nlay, axis=0).astype(np.int32)

    top2d[id2d == 0] = 0.0
    botm3d[:, id2d == 0] = np.nan
    hk3d[:, id2d == 0] = 0.0
    drain_frac2d[id2d <= 0] = 0.0
    drain_elev2d[id2d <= 0] = np.nan

    strt2d = np.where(np.isfinite(strt2d_raw), strt2d_raw, top2d)
    strt = np.repeat(strt2d[np.newaxis, :, :], nlay, axis=0).astype("float32")
    strt[idomain == 0] = 0.0

    print("Using nlay =", nlay, "to match HK bands")
    print("Active after 5-layer build:", int(np.count_nonzero(id2d)))

    total_thk = top2d - botm3d[-1]
    vals = total_thk[id2d > 0]
    vals = vals[np.isfinite(vals)]
    if vals.size > 0:
        print("\nTotal thickness summary (m)")
        print("  min :", vals.min())
        print("  p1  :", np.percentile(vals, 1))
        print("  p5  :", np.percentile(vals, 5))
        print("  p50 :", np.percentile(vals, 50))
        print("  p95 :", np.percentile(vals, 95))
        print("  p99 :", np.percentile(vals, 99))
        print("  max :", vals.max())

In [ ]:


def compute_exact_layer_thickness(top2d, botm3d, id2d=None):
    """
    Compute exact thickness of each layer from top and bottom surfaces.

    Parameters
    ----------
    top2d : ndarray of shape (nrow, ncol)
        Model top elevation.
    botm3d : ndarray of shape (nlay, nrow, ncol)
        Bottom elevation of each layer.
    id2d : ndarray of shape (nrow, ncol), optional
        Active-domain mask. Active cells should be 1, inactive 0.

    Returns
    -------
    thickness : ndarray of shape (nlay, nrow, ncol)
        Exact layer thickness array.
    """
    nlay = botm3d.shape[0]
    thickness = np.full_like(botm3d, np.nan, dtype=float)

    # Layer 1
    thickness[0] = top2d - botm3d[0]

    # Layers 2+
    for k in range(1, nlay):
        thickness[k] = botm3d[k - 1] - botm3d[k]

    if id2d is not None:
        inactive = (id2d <= 0)
        thickness[:, inactive] = np.nan

    return thickness


def print_exact_thickness_summary(thickness, id2d=None, label=""):
    """
    Print summary statistics for exact layer thickness.

    Parameters
    ----------
    thickness : ndarray
        Exact layer thickness array.
    id2d : ndarray, optional
        Active-domain mask.
    label : str, optional
        Label for summary header.
    """
    if label:
        print(f"\n{'='*40}\n{label}\n{'='*40}")

    nlay = thickness.shape[0]

    for k in range(nlay):
        arr = thickness[k]

        if id2d is not None:
            vals = arr[id2d > 0]
        else:
            vals = arr[np.isfinite(arr)]

        vals = vals[np.isfinite(vals)]

        if vals.size == 0:
            print(f"\nLayer {k+1}: no valid cells")
            continue

        print(f"\nLayer {k+1}")
        print(f"  min  : {vals.min():.3f}")
        print(f"  p5   : {np.percentile(vals, 5):.3f}")
        print(f"  p50  : {np.percentile(vals, 50):.3f}")
        print(f"  mean : {vals.mean():.3f}")
        print(f"  p95  : {np.percentile(vals, 95):.3f}")
        print(f"  max  : {vals.max():.3f}")
        print(f"  <=0  : {np.sum(vals <= 0)}")

    total_thk = np.nansum(thickness, axis=0)
    if id2d is not None:
        vals = total_thk[id2d > 0]
    else:
        vals = total_thk[np.isfinite(total_thk)]

    vals = vals[np.isfinite(vals)]
    if vals.size > 0:
        print("\nTotal model thickness")
        print(f"  min  : {vals.min():.3f}")
        print(f"  p5   : {np.percentile(vals, 5):.3f}")
        print(f"  p50  : {np.percentile(vals, 50):.3f}")
        print(f"  mean : {vals.mean():.3f}")
        print(f"  p95  : {np.percentile(vals, 95):.3f}")
        print(f"  max  : {vals.max():.3f}")


def plot_exact_thickness_maps(thickness, id2d=None, title_prefix="Layer", same_scale=True):
    """
    Plot exact thickness maps for all layers.

    Parameters
    ----------
    thickness : ndarray of shape (nlay, nrow, ncol)
        Exact layer thickness array.
    id2d : ndarray, optional
        Active-domain mask.
    title_prefix : str, optional
        Prefix for plot titles.
    same_scale : bool, optional
        If True, use one shared color scale across all layers.
    """
    nlay = thickness.shape[0]

    # Mask inactive cells
    plot_arr = thickness.copy()
    if id2d is not None:
        plot_arr[:, id2d <= 0] = np.nan

    # Global scale
    if same_scale:
        finite_vals = plot_arr[np.isfinite(plot_arr)]
        vmin = np.nanmin(finite_vals)
        vmax = np.nanmax(finite_vals)
    else:
        vmin, vmax = None, None

    ncols = min(3, nlay)
    nrows = int(np.ceil(nlay / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(6*ncols, 5*nrows))
    axes = np.atleast_1d(axes).ravel()

    for k in range(nlay):
        ax = axes[k]
        arr = plot_arr[k]

        if same_scale:
            im = ax.imshow(arr, vmin=vmin, vmax=vmax)
        else:
            im = ax.imshow(arr)

        vals = arr[np.isfinite(arr)]
        if vals.size > 0:
            ax.set_title(
                f"{title_prefix} {k+1} exact thickness (m)\n"
                f"min={vals.min():.1f}, mean={vals.mean():.1f}, max={vals.max():.1f}"
            )
        else:
            ax.set_title(f"{title_prefix} {k+1} exact thickness (m)")

        ax.set_xlabel("Column")
        ax.set_ylabel("Row")
        cbar = plt.colorbar(im, ax=ax, shrink=0.85)
        cbar.set_label("Thickness (m)")

    for k in range(nlay, len(axes)):
        axes[k].axis("off")

    plt.tight_layout()
    plt.show()


def print_sample_active_values(thickness, id2d, n_cells=10):
    """
    Print exact thickness values for a few active cells.

    Parameters
    ----------
    thickness : ndarray
        Exact thickness array.
    id2d : ndarray
        Active-domain mask.
    n_cells : int, optional
        Number of active cells to print.
    """
    rc = np.argwhere(id2d > 0)

    print("\nSample exact thickness values at active cells")
    print("-" * 50)

    for idx, (i, j) in enumerate(rc[:n_cells], start=1):
        vals = [thickness[k, i, j] for k in range(thickness.shape[0])]
        vals_str = ", ".join([f"L{k+1}={v:.2f}" for k, v in enumerate(vals)])
        print(f"Cell {idx}: row={i}, col={j} -> {vals_str}")


# =========================================================
# A) FINAL MODEL THICKNESS (after any 5-layer rebuild)
# =========================================================
thickness_final = compute_exact_layer_thickness(top2d, botm3d, id2d=id2d)
print_exact_thickness_summary(thickness_final, id2d=id2d, label="FINAL MODEL LAYER THICKNESS")
plot_exact_thickness_maps(thickness_final, id2d=id2d, title_prefix="Final layer", same_scale=True)
print_sample_active_values(thickness_final, id2d=id2d, n_cells=10)


# =========================================================
# B) ORIGINAL INPUT THICKNESS (before 5-layer rebuild)
# Only works if you saved top2d_input, botm3d_input, id2d_input
# =========================================================
if "botm3d_input" in globals():
    thickness_input = compute_exact_layer_thickness(top2d_input, botm3d_input, id2d=id2d_input)
    print_exact_thickness_summary(thickness_input, id2d=id2d_input, label="ORIGINAL INPUT BOTM THICKNESS")
    plot_exact_thickness_maps(thickness_input, id2d=id2d_input, title_prefix="Original input layer", same_scale=True)
    print_sample_active_values(thickness_input, id2d=id2d_input, n_cells=10)

In [ ]:
# =========================================================
# VERTICAL HYDRAULIC CONDUCTIVITY (anisotropy)
# Kv = Kh / KV_ANISOTROPY_RATIO  per layer band
# Each layer's Kv uses the same ratio applied to its own Kh,
# consistent with the 5-band HK structure.
# =========================================================

KV_ANISOTROPY_RATIO = 10.0    # Kv = Kh / 10 for all layers

k33_3d = hk3d / KV_ANISOTROPY_RATIO

# enforce same floor/ceil as horizontal K, inactive cells = 0
k33_min = 1e-8 / KV_ANISOTROPY_RATIO   # slightly below hk_min/10
k33_max = 1e4  / KV_ANISOTROPY_RATIO

m = (id2d == 1)
for k in range(nlay):
    k33_3d[k, m]  = np.clip(k33_3d[k, m], k33_min, k33_max)
    k33_3d[k, ~m] = 0.0

k33_3d = k33_3d.astype("float32")

print("k33 (Kv) shape:", k33_3d.shape)
print("k33 Layer 1 min/max (active):",
      float(k33_3d[0, m].min()), float(k33_3d[0, m].max()))
print("Ratio check (Kh/Kv):",
      round(float(hk3d[0, m].mean() / k33_3d[0, m].mean()), 2),
      "— should be", KV_ANISOTROPY_RATIO)

## 7) Build the Great Lakes polygon mask and reactivate valid lake cells

The basin boundary can exclude open-water interiors depending on how the original polygon was constructed.
This section builds the lake polygon mask directly from `GreatLakes.shp`, checks thickness validity,
and reactivates lake cells that have valid model geometry.

`lake_mask_full` = all lake polygon cells on the model grid  
`lake_mask_2d`   = active lake cells on the final model grid


In [ ]:
import geopandas as gpd
import rasterio as rio
from rasterio.features import rasterize
import numpy as np

pathLakePoly = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Lakes\GreatLakes.shp"

def build_full_lake_mask(path_lake_poly, template_tif):
    gdf = gpd.read_file(path_lake_poly)

    with rio.open(template_tif) as tmp:
        lake_mask = rasterize(
            [(geom, 1) for geom in gdf.geometry if geom is not None and not geom.is_empty],
            out_shape=(tmp.height, tmp.width),
            transform=tmp.transform,
            fill=0,
            dtype="uint8",
            all_touched=False
        ).astype(bool)

    return lake_mask

# full lake polygon on model grid
lake_mask_full = build_full_lake_mask(pathLakePoly, template_tif)

print("Total lake polygon cells:", int(lake_mask_full.sum()))
print("Currently active lake cells:", int(np.sum((id2d == 1) & lake_mask_full)))
print("Currently inactive lake cells:", int(np.sum((id2d == 0) & lake_mask_full)))

# keep only lake cells with valid thickness in all layers
lake_keep_mask = lake_mask_full.copy()

thk1 = top2d - botm3d[0]
lake_keep_mask &= np.isfinite(thk1) & (thk1 > 0)

for k in range(1, nlay):
    thk = botm3d[k - 1] - botm3d[k]
    lake_keep_mask &= np.isfinite(thk) & (thk > 0)

print("Lake cells with valid thickness:", int(lake_keep_mask.sum()))

# reactivate them
id2d = id2d.copy()
id2d[lake_keep_mask] = 1

idomain = np.repeat(id2d[np.newaxis, :, :], nlay, axis=0).astype(np.int32)

# this is the ACTIVE lake mask for later steps
lake_mask_2d = lake_mask_full & (id2d == 1)

print("Active lake cells after reactivation:", int(np.sum((id2d == 1) & lake_mask_full)))
print("Inactive lake cells after reactivation:", int(np.sum((id2d == 0) & lake_mask_full)))
print("Active cells total after reactivation:", int(np.sum(id2d == 1)))

## 8) Finalize revised starting heads

This revises the starting heads so they are:
- below land surface
- above the cell bottom
- valid only in active cells


In [ ]:
# =========================================================
# PART 8 — REVISED STARTING HEADS
# =========================================================

TOP_BUFFER = 0.5               # keep heads slightly below land surface
MIN_ABOVE_BOT = 2.0            # minimum head above cell bottom
MIN_SAT_FRAC = 0.30            # at least 30% of layer thickness above bottom

# layer 1 thickness
thk1 = np.maximum(top2d - botm3d[0], 0.1)

# raw starting-head target:
# use interpolated starting head if available, otherwise top - 2 m
raw_strt1 = np.where(np.isfinite(strt2d_raw), strt2d_raw, top2d - 2.0)

# cap below land surface
strt1 = np.minimum(raw_strt1, top2d - TOP_BUFFER)

# push heads upward if too close to bottom
bottom_floor = botm3d[0] + np.maximum(MIN_ABOVE_BOT, MIN_SAT_FRAC * thk1)
strt1 = np.maximum(strt1, bottom_floor)

# keep only active top-layer cells
strt1 = np.where(idomain[0] > 0, strt1, 0.0)

# OPTIONAL: enforce GHB stage on GHB cells for first period
if USE_GHB and "ghb_cells_df" in globals() and "monthly_stages_model" in globals():
    model_months = build_model_months("2000-01-01", len(perioddata_run))
    first_dt = model_months[0]
    stage_lookup = monthly_stages_model.loc[first_dt].to_dict()

    for r in ghb_cells_df.itertuples(index=False):
        i = int(r.i)
        j = int(r.j)
        stg = float(stage_lookup[r.stage_name])
        strt1[i, j] = np.clip(
            stg,
            botm3d[0, i, j] + MIN_ABOVE_BOT,
            top2d[i, j] - TOP_BUFFER
        )

# build 3D strt
strt3d = np.zeros((nlay, nrow, ncol), dtype="float32")
strt3d[0] = strt1

for k in range(1, nlay):
    lay_thk = np.maximum(botm3d[k - 1] - botm3d[k], 0.1)
    lower_floor = botm3d[k] + np.maximum(1.0, 0.20 * lay_thk)
    strt3d[k] = np.maximum(
        np.minimum(strt3d[k - 1] - 0.5, botm3d[k - 1] - 0.5),
        lower_floor
    )
    strt3d[k] = np.where(idomain[k] > 0, strt3d[k], 0.0)

strt = strt3d.copy()

# diagnostics
active = idomain[0] > 0
diff_bottom = strt[0][active] - botm3d[0][active]
diff_top = top2d[active] - strt[0][active]

print("REVISED STRT - BOT min/max:", np.nanmin(diff_bottom), np.nanmax(diff_bottom))
print("REVISED TOP - STRT min/max:", np.nanmin(diff_top), np.nanmax(diff_top))
print("Cells within 0.5 m of bottom:", np.sum(diff_bottom <= 0.5))
print("Percent within 0.5 m of bottom:", 100 * np.mean(diff_bottom <= 0.5))
print("Cells above top:", np.sum(strt[0][active] > top2d[active]))

## 9) Build the structured model grid and set CHD behavior

The current workflow uses **no CHD package**.
The model grid and `GridIntersect` object are created here for later package construction.


In [ ]:
structuredGrid = flopy.discretization.StructuredGrid(
    nlay=nlay, nrow=nrow, ncol=ncol,
    delr=delr, delc=delc,
    xoff=xorigin, yoff=yorigin, angrot=0.0
)
ix = make_gridintersect(structuredGrid)
print("GridIntersect ready.")


In [ ]:
# No CHD in this model
USE_CHD = False
chd_rec = []
dfChd = pd.DataFrame(columns=["row", "col", "head"])

print("Skipping CHD. chd_rec is empty.")

## 10) Time discretization and recharge

This builds monthly stress periods and recharge arrays from the indexed recharge inputs.
After recharge is built, recharge is forced to zero over **active lake cells**.


In [ ]:
# use boundary CRS as the destination CRS for reprojection
grid_crs_wkt = gpd.read_file(boundary_shp).crs.to_wkt()
print("Destination CRS loaded from boundary.")
print(grid_crs_wkt[:120], "...")

In [ ]:
# ---------------------------------------------------------
# BUILD MODEL MONTHS / PERIODS
# ---------------------------------------------------------
perioddata, months = build_monthly_perioddata(START_DATE, END_DATE)
nper = len(perioddata)
print("Stress periods in full run:", nper)

# ---------------------------------------------------------
# INDEX NLDAS FILES
# ---------------------------------------------------------
df_nc = index_blend_qsb_monthlies(nldas_root)

# force all dates to month-start timestamps so matching is consistent
df_nc = df_nc.copy()
df_nc["date"] = (
    pd.DatetimeIndex(df_nc["date"])
    .to_period("M")
    .to_timestamp(how="start")
)

expected = (
    pd.date_range(START_DATE, END_DATE, freq="MS")
    .to_period("M")
    .to_timestamp(how="start")
)

missing = expected.difference(df_nc["date"])
print("Missing NLDAS months:", len(missing))
if len(missing) > 0:
    print("First missing months:", list(missing[:10]))

# ---------------------------------------------------------
# SUBSET RUN PERIODS
# ---------------------------------------------------------
if NPER_TEST is not None:
    perioddata_run = perioddata[:NPER_TEST]
    months_run = months[:NPER_TEST]
else:
    perioddata_run = perioddata
    months_run = months

months_run = (
    pd.DatetimeIndex(months_run)
    .to_period("M")
    .to_timestamp(how="start")
)

print("Running nper =", len(perioddata_run))
print("Run dates:", months_run[0], "to", months_run[-1])

df_nc_run = df_nc[df_nc["date"].isin(months_run)].copy()

# ---------------------------------------------------------
# GET DESTINATION CRS FROM BOUNDARY, NOT template_tif
# this avoids relying on tmp.crs inside rasterio
# ---------------------------------------------------------
grid_crs_wkt = gpd.read_file(boundary_shp).crs.to_wkt()

# ---------------------------------------------------------
# BUILD RECHARGE STRESS-PERIOD ARRAYS
# ---------------------------------------------------------
rch_spd = build_rch_spd_from_index(
    df_nc=df_nc_run,
    months_run=months_run,
    template_tif=template_tif,
    id2d=id2d,
    var=NLDAS_VAR,
    dst_crs_wkt=grid_crs_wkt,   # <-- new
)

print("Recharge stress periods built:", len(rch_spd))
first_key = sorted(rch_spd.keys())[0]
print(
    "Recharge sample min/max:",
    float(np.nanmin(rch_spd[first_key])),
    float(np.nanmax(rch_spd[first_key]))
)

# ---------------------------------------------------------
# STEADY-STATE RECHARGE ARRAY
# note: this computes it, but does NOT prepend a new SS period
# unless you uncomment the block below
# ---------------------------------------------------------
rch_all_arr = np.array([rch_spd[p] for p in range(len(rch_spd))], dtype=float)
rch_mean_ss = np.nanmean(rch_all_arr, axis=0)
rch_mean_ss[id2d <= 0] = 0.0
rch_mean_ss[~np.isfinite(rch_mean_ss)] = 0.0

# steady-state period: 10 years, 1 timestep, multiplier 1.0
# SS_YEARS = 1
# ss_period = (365.0 * SS_YEARS, 1, 1.0)

# prepend SS period to perioddata_run
# perioddata_run = [ss_period] + list(perioddata_run)

# shift all existing rch_spd keys up by 1, add SS period as key 0
# rch_spd_new = {0: rch_mean_ss}
# for old_per, arr in rch_spd.items():
#     rch_spd_new[old_per + 1] = arr
# rch_spd = rch_spd_new

print("\n--- Recharge build complete ---")
print(f"Total stress periods in run: {len(perioddata_run)}")
print(f"rch_spd periods: {len(rch_spd)}")
print(
    f"Recharge SS-style mean min/max: "
    f"{float(np.nanmin(rch_mean_ss[id2d > 0])):.2e} / "
    f"{float(np.nanmax(rch_mean_ss[id2d > 0])):.2e} m/day"
)

In [ ]:
# ---------------------------------------------------------
# ZERO RECHARGE OVER ACTIVE GREAT LAKES CELLS
# ---------------------------------------------------------
for per in rch_spd:
    arr = np.asarray(rch_spd[per], dtype=float).copy()
    arr[lake_mask_2d] = 0.0
    rch_spd[per] = arr.astype("float32")

print("Recharge over active lake cells set to 0 for all periods.")


In [ ]:
# =========================================================
# RECHARGE DIAGNOSTIC — check if Qsb is realistic
# Expected for Great Lakes Basin: 100–300 mm/yr
# =========================================================
import numpy as np

active_land = (idomain[0] > 0)
if "lake_mask_2d" in globals():
    active_land = active_land & (~lake_mask_2d)

rch_vals_mmyr = []
for p in sorted(rch_spd.keys()):
    arr = np.asarray(rch_spd[p], dtype=float)
    vals = arr[active_land]
    vals = vals[np.isfinite(vals)]
    if vals.size > 0:
        rch_vals_mmyr.append(float(np.nanmean(vals)) * 365.25 * 1000)

mean_rch_mmyr = float(np.mean(rch_vals_mmyr)) if rch_vals_mmyr else np.nan

print("=" * 50)
print("RECHARGE MAGNITUDE CHECK")
print("=" * 50)
print(f"  Mean recharge across periods : {mean_rch_mmyr:.1f} mm/yr")
print(f"  Min  period mean             : {min(rch_vals_mmyr):.1f} mm/yr")
print(f"  Max  period mean             : {max(rch_vals_mmyr):.1f} mm/yr")
print()

if mean_rch_mmyr > 500:
    print("  ❌ FAIL — recharge >> 500 mm/yr")
    print("     Qsb is overestimating recharge.")
    print("     ACTION: apply a multiplier (see below) or switch NLDAS variable.")
    # --- apply a scaling factor ---
    RCH_SCALE = 0.3   # <-- adjust this (0.1 to 0.5 typical)
    for p in rch_spd:
        rch_spd[p] = (np.asarray(rch_spd[p], dtype=float) * RCH_SCALE).astype("float32")
    print(f"  ✅ Applied RCH_SCALE = {RCH_SCALE} to all periods")
    print(f"  New mean recharge: {mean_rch_mmyr * RCH_SCALE:.1f} mm/yr")

elif mean_rch_mmyr > 300:
    print("  ⚠️  WARNING — recharge > 300 mm/yr, on the high side")
    print("     Consider applying RCH_SCALE = 0.5")

else:
    print("  ✅ PASS — recharge in expected range (100–300 mm/yr)")

### 10a) Recharge check over lake cells

This diagnostic confirms that recharge is zero over the active lake mask.


In [ ]:

lake_rch_max = np.nan
lake_rch_any_nonzero = False

if "lake_mask_2d" not in globals():
    print("lake_mask_2d not found")
else:
    max_list = []
    for p in rch_spd:
        arr = np.asarray(rch_spd[p], dtype=float)

        if arr.shape != lake_mask_2d.shape:
            print(f"Skipping period {p}: shape mismatch {arr.shape} vs {lake_mask_2d.shape}")
            continue

        vals = arr[lake_mask_2d]
        vals = vals[np.isfinite(vals)]

        if vals.size > 0:
            max_list.append(vals.max())
            lake_rch_any_nonzero = lake_rch_any_nonzero or np.any(vals > 0)

    if len(max_list) > 0:
        lake_rch_max = max(max_list)

print("Max recharge over lake cells:", lake_rch_max)
print("Any non-zero recharge over lake cells:", lake_rch_any_nonzero)

## 11) Build the General-Head Boundary (GHB)

Order matters here:
1. define GHB settings
2. rebuild the GHB cell table on the **current model grid**
3. build transient GHB stress-period data
4. run post-GHB diagnostics


In [ ]:
# =========================================================
# PART 9.5 — GHB SOURCE SETTINGS
# =========================================================

# This must be the ORIGINAL GIS feature used to define the GHB footprint
# (for example: shoreline band polygon / lake band polygon / lake-edge polygon)
# DO NOT use the old GreatLakes_GHB_cells.csv here
pathInputGHBFeature = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Lakes\GreatLakes_buffer10km.shp"

# field in that GIS layer that contains lake names
lake_name_field = "lake_name"   # change if needed

# output files
OUT_GHB_TABLE = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Lakes\GreatLakes_GHB_cells_currentGrid.csv"
OUT_STAGE_TABLE = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Lakes\GreatLakes_stage_monthly_for_model.csv"

# conductance assumptions
GHB_BED_THICKNESS_M = 1.0
GHB_KV_DIVISOR = 10.0

print("GHB source feature:", pathInputGHBFeature)
print("Lake name field:", lake_name_field)
print("OUT_GHB_TABLE:", OUT_GHB_TABLE)
print("OUT_STAGE_TABLE:", OUT_STAGE_TABLE)
print("GHB_BED_THICKNESS_M:", GHB_BED_THICKNESS_M)
print("GHB_KV_DIVISOR:", GHB_KV_DIVISOR)

In [ ]:
# =========================================================
# PART 9.6 — REBUILD GHB CELL TABLE ON CURRENT GRID
# =========================================================

import numpy as np
import pandas as pd
import flopy

# ---------------------------------------------------------
# build current model grid + GridIntersect
# ---------------------------------------------------------
structuredGrid = flopy.discretization.StructuredGrid(
    nlay=nlay,
    nrow=nrow,
    ncol=ncol,
    delr=delr,
    delc=delc,
    xoff=xorigin,
    yoff=yorigin,
    angrot=0.0,
)

ix = make_gridintersect(structuredGrid)

# ---------------------------------------------------------
# intersect current GHB source feature with current grid
# ---------------------------------------------------------
ghb_src = intersect_grid_feature(
    ix=ix,
    pathFeature=pathInputGHBFeature,
    lay=0,
    addFields=[lake_name_field],
    grid_crs=grid_crs,
)

if ghb_src.empty:
    raise ValueError("No intersections found. Check pathInputGHBFeature, lake_name_field, and CRS.")

print("Raw GHB intersections:", len(ghb_src))
print("Columns:", ghb_src.columns.tolist())

# ---------------------------------------------------------
# standardize lake names
# ---------------------------------------------------------
if lake_name_field not in ghb_src.columns:
    raise ValueError(f"Field '{lake_name_field}' not found in source feature.")

ghb_src = ghb_src.copy()
ghb_src["lake_name"] = ghb_src[lake_name_field].astype(str).str.strip()

valid_lakes = {"Superior", "Michigan", "Huron", "Erie", "Ontario"}
ghb_src = ghb_src[ghb_src["lake_name"].isin(valid_lakes)].copy()

if ghb_src.empty:
    raise ValueError("No valid Great Lakes names found in source feature.")

# ---------------------------------------------------------
# get overlap area
# ---------------------------------------------------------
if "areas" in ghb_src.columns:
    ghb_src["overlap_m2"] = pd.to_numeric(ghb_src["areas"], errors="coerce")
elif "area" in ghb_src.columns:
    ghb_src["overlap_m2"] = pd.to_numeric(ghb_src["area"], errors="coerce")
else:
    raise ValueError(
        "No polygon overlap area column found. "
        "Your GHB source feature should be a polygon/band feature, not a line."
    )

ghb_src = ghb_src[ghb_src["overlap_m2"].notna() & (ghb_src["overlap_m2"] > 0)].copy()

# ---------------------------------------------------------
# assign current grid indices
# ---------------------------------------------------------
ghb_src["i"] = ghb_src["row"].astype(int)
ghb_src["j"] = ghb_src["col"].astype(int)
ghb_src["k"] = 0

# ---------------------------------------------------------
# keep only active top-layer cells
# ---------------------------------------------------------
if idomain.ndim == 3:
    active_top = idomain[0] > 0
else:
    active_top = idomain > 0

rr = ghb_src["i"].to_numpy(dtype=int)
cc = ghb_src["j"].to_numpy(dtype=int)

keep = (
    (rr >= 0) & (rr < nrow) &
    (cc >= 0) & (cc < ncol) &
    active_top[rr, cc]
)

ghb_src = ghb_src.loc[keep].copy()

if ghb_src.empty:
    raise ValueError("All GHB cells were removed by current idomain.")

# ---------------------------------------------------------
# hydraulic properties from current grid
# ---------------------------------------------------------
rr = ghb_src["i"].to_numpy(dtype=int)
cc = ghb_src["j"].to_numpy(dtype=int)

ghb_src["cell_area_m2"] = delr[cc] * delc[rr]
ghb_src["hk_mday"] = hk3d[0, rr, cc].astype(float)
ghb_src["kv_mday"] = np.maximum(ghb_src["hk_mday"] / GHB_KV_DIVISOR, 1e-8)

# conductance
ghb_src["cond"] = (
    ghb_src["kv_mday"] * ghb_src["overlap_m2"] / float(GHB_BED_THICKNESS_M)
).astype(float)

# ---------------------------------------------------------
# map stage names to stage-table columns
# ---------------------------------------------------------
ghb_src["stage_name"] = ghb_src["lake_name"].replace({
    "Superior": "Superior",
    "Michigan": "Michigan-Huron",
    "Huron": "Michigan-Huron",
    "Erie": "Erie",
    "Ontario": "Ontario",
})

ghb_src["bname"] = ghb_src.apply(
    lambda r: f"{r['lake_name']}_{int(r['i'])}_{int(r['j'])}",
    axis=1,
)

# ---------------------------------------------------------
# collapse duplicates per cell
# ---------------------------------------------------------
ghb_cells_df = (
    ghb_src.groupby(["lake_name", "stage_name", "k", "i", "j", "bname"], as_index=False)
    .agg(
        overlap_m2=("overlap_m2", "sum"),
        cell_area_m2=("cell_area_m2", "first"),
        hk_mday=("hk_mday", "first"),
        kv_mday=("kv_mday", "first"),
        cond=("cond", "sum"),
    )
)

ghb_cells_df = ghb_cells_df[ghb_cells_df["cond"] > 0].copy()

# save
ghb_cells_df.to_csv(OUT_GHB_TABLE, index=False)

print("Saved rebuilt GHB table:")
print(OUT_GHB_TABLE)
print()
print("Rebuilt GHB lakes:", sorted(ghb_cells_df["lake_name"].unique()))
print(ghb_cells_df["lake_name"].value_counts().sort_index())
print()
print("Rebuilt i range:", int(ghb_cells_df["i"].min()), int(ghb_cells_df["i"].max()))
print("Rebuilt j range:", int(ghb_cells_df["j"].min()), int(ghb_cells_df["j"].max()))
print("Current grid shape:", (nrow, ncol))

In [ ]:
# =========================================================
# PART 10 — BUILD TRANSIENT GHB STRESS-PERIOD DATA
# =========================================================

import numpy as np
import pandas as pd

# ---------------------------------------------------------
# read rebuilt GHB cells + monthly stage table
# ---------------------------------------------------------
ghb_cells_df = pd.read_csv(OUT_GHB_TABLE, low_memory=False)
monthly_stages_model = pd.read_csv(OUT_STAGE_TABLE, index_col=0, parse_dates=True)

# normalize monthly stage index to month-start
monthly_stages_model.index = (
    pd.DatetimeIndex(monthly_stages_model.index)
    .to_period("M")
    .to_timestamp(how="start")
)

# use same months as TDIS — subtract 1 because period 0 is SS not a calendar month
model_months = build_model_months("2000-01-01", len(perioddata_run) - 1)

# ---------------------------------------------------------
# standardize lake / stage names
# ---------------------------------------------------------
ghb_cells_df["lake_name"] = ghb_cells_df["lake_name"].astype(str).str.strip()
ghb_cells_df["stage_name"] = ghb_cells_df["lake_name"].replace({
    "Superior": "Superior",
    "Michigan": "Michigan-Huron",
    "Huron": "Michigan-Huron",
    "Erie": "Erie",
    "Ontario": "Ontario",
})

# ---------------------------------------------------------
# check stage columns exist
# ---------------------------------------------------------
needed_stage_names = sorted(set(ghb_cells_df["stage_name"]))
missing_stage_names = [nm for nm in needed_stage_names
                       if nm not in monthly_stages_model.columns]
if missing_stage_names:
    raise ValueError(f"Missing stage columns in stage table: {missing_stage_names}")

# ---------------------------------------------------------
# check month coverage
# ---------------------------------------------------------
missing_months = [dt for dt in model_months
                  if dt not in monthly_stages_model.index]
if missing_months:
    raise ValueError(
        f"Stage table is missing {len(missing_months)} model months. "
        f"First missing month: {missing_months[0]}"
    )

# ---------------------------------------------------------
# choose GHB layer k using minimum lake stage over modeled period
# ---------------------------------------------------------
stage_floor_by_name = monthly_stages_model.loc[
    model_months, needed_stage_names
].min()

print("Minimum stage by lake used for GHB layer assignment:")
print(stage_floor_by_name)

ghb_cells_df = assign_ghb_k_from_stage_floor(
    ghb_cells_df=ghb_cells_df,
    idomain=idomain,
    botm3d=botm3d,
    stage_floor_by_name=stage_floor_by_name,
    stage_margin=0.05,
)

print("Prepared GHB cells after stage-aware update:", len(ghb_cells_df))
print("GHB lakes:", sorted(ghb_cells_df["lake_name"].unique()))
print("Per-lake counts after k assignment:")
print(ghb_cells_df["lake_name"].value_counts().sort_index())

# ---------------------------------------------------------
# sanity check: every GHB cell bottom must be below stage floor
# ---------------------------------------------------------
bad = []
for r in ghb_cells_df.itertuples(index=False):
    stg = float(stage_floor_by_name[r.stage_name])
    btm = float(botm3d[int(r.k), int(r.i), int(r.j)])
    if not (stg > btm):
        bad.append((r.stage_name, int(r.k), int(r.i), int(r.j), stg, btm))

print("Bad GHB cells after reassignment:", len(bad))
if len(bad) > 0:
    print("First 10 bad cells:")
    print(bad[:10])
    raise ValueError("Some GHB cells still have stage <= cell bottom")

# ---------------------------------------------------------
# REDUCE GHB CONDUCTANCE
# ---------------------------------------------------------
GHB_COND_MULT = 0.1   # reduced from 0.5 — limits shoreline leakage
ghb_cells_df["cond"] = ghb_cells_df["cond"] * GHB_COND_MULT

print(f"GHB conductance multiplied by {GHB_COND_MULT}")
print(f"GHB cond min : {ghb_cells_df['cond'].min():.3e}")
print(f"GHB cond max : {ghb_cells_df['cond'].max():.3e}")
print(f"GHB cond mean: {ghb_cells_df['cond'].mean():.3e}")

# ---------------------------------------------------------
# ADD top_cap to ghb_cells_df
# ---------------------------------------------------------
rr = ghb_cells_df["i"].to_numpy(dtype=int)
cc = ghb_cells_df["j"].to_numpy(dtype=int)
ghb_cells_df["top_cap"] = top2d[rr, cc]

print("top_cap min/max:",
      ghb_cells_df["top_cap"].min(),
      ghb_cells_df["top_cap"].max())

print("Cells where stage_floor > top_cap:",
      int(np.sum(
          stage_floor_by_name[ghb_cells_df["stage_name"].values].values
          > ghb_cells_df["top_cap"].values
      )))

# ---------------------------------------------------------
# BUILD TRANSIENT ghb_spd — one entry per modeled month
# temporary keys 0..N-1 for transient months only
# ---------------------------------------------------------
STAGE_CAP_OFFSET = 0.10

ghb_spd_transient = {}
for iper, dt in enumerate(model_months):
    recs = []
    for r in ghb_cells_df.itertuples(index=False):
        raw_stage = float(monthly_stages_model.loc[dt, r.stage_name])
        stage = min(raw_stage, float(r.top_cap) - STAGE_CAP_OFFSET)
        recs.append((
            (int(r.k), int(r.i), int(r.j)),
            stage,
            float(r.cond),
            str(r.bname),
        ))
    ghb_spd_transient[iper] = recs

print("\nGHB stress periods (transient only):", len(ghb_spd_transient))
print("GHB maxbound:", max(len(v) for v in ghb_spd_transient.values()))
print("Same GHB count each period?",
      all(len(v) == len(ghb_spd_transient[0]) for v in ghb_spd_transient.values()))

n_capped = sum(
    1 for r in ghb_cells_df.itertuples(index=False)
    if float(monthly_stages_model.loc[model_months[0], r.stage_name])
       > float(r.top_cap) - STAGE_CAP_OFFSET
)
print(f"GHB cells with stage capped in first transient month: {n_capped:,} of {len(ghb_cells_df):,}")

# ---------------------------------------------------------
# SHIFT ghb_spd TO ACCOUNT FOR STEADY-STATE PERIOD 0
# Period 0 = SS spin-up -> use first transient month stage
# Periods 1..N = transient months shifted by +1
# ---------------------------------------------------------
ghb_spd = {0: ghb_spd_transient[0]}

for old_per, recs in ghb_spd_transient.items():
    ghb_spd[old_per + 1] = recs

print("\nGHB stress periods after SS shift:", len(ghb_spd))
print("ghb_spd keys (first 5):", sorted(ghb_spd.keys())[:5])
print("ghb_spd keys (last):", sorted(ghb_spd.keys())[-1])

# ---------------------------------------------------------
# FINAL CONSISTENCY CHECK
# ---------------------------------------------------------
print("\n--- Period count consistency check ---")
print(f"perioddata_run: {len(perioddata_run)}")
print(f"ghb_spd:        {len(ghb_spd)}")

if len(ghb_spd) != len(perioddata_run):
    raise ValueError(
        f"ghb_spd has {len(ghb_spd)} periods but "
        f"perioddata_run has {len(perioddata_run)}. "
        f"They must match."
    )

print("✅ Period counts match")

In [ ]:
# =========================================================
# POST-GHB CHECKS
# Run this AFTER Part 10
# =========================================================
# ---------------------------------------------------------
# sanity check: every assigned GHB cell must have bottom below that lake's minimum stage
# ---------------------------------------------------------
bad = []
for r in ghb_cells_df.itertuples(index=False):
    stg = float(stage_floor_by_name[r.stage_name])
    btm = float(botm3d[int(r.k), int(r.i), int(r.j)])
    if not (stg > btm):
        bad.append((r.stage_name, int(r.k), int(r.i), int(r.j), stg, btm))

print("Bad GHB cells after reassignment:", len(bad))
if len(bad) > 0:
    print("First 10 bad cells:")
    print(bad[:10])
    raise ValueError("Some GHB cells still have stage <= cell bottom")

# ---------------------------------------------------------
# Build ring mask directly from rebuilt GHB cells
# ---------------------------------------------------------
ring_mask_2d = np.zeros((nrow, ncol), dtype=bool)
for r in ghb_cells_df.itertuples(index=False):
    i, j = int(r.i), int(r.j)
    if (0 <= i < nrow) and (0 <= j < ncol):
        ring_mask_2d[i, j] = True

print(f"Ring cells: {int(ring_mask_2d.sum()):,}")

# ---------------------------------------------------------
# Check recharge over ring across all periods
# ---------------------------------------------------------
if ring_mask_2d.sum() == 0:
    print("No ring cells found on current grid; skipping recharge-over-ring check.")
    ring_rch_max = np.nan
    ring_rch_any_nonzero = False
else:
    ring_max_list = []
    ring_rch_any_nonzero = False

    for p in rch_spd:
        arr = np.asarray(rch_spd[p], dtype=float)

        if arr.shape != ring_mask_2d.shape:
            print(f"Skipping period {p}: recharge shape {arr.shape} does not match grid {ring_mask_2d.shape}")
            continue

        vals = arr[ring_mask_2d]
        vals = vals[np.isfinite(vals)]

        if vals.size > 0:
            ring_max_list.append(vals.max())
            ring_rch_any_nonzero = ring_rch_any_nonzero or np.any(vals > 0)

    ring_rch_max = max(ring_max_list) if len(ring_max_list) > 0 else np.nan

print(f"Max recharge over ring across all periods: {ring_rch_max}")
print(f"Any non-zero recharge over ring: {ring_rch_any_nonzero}")

## 12) Prepare and build drains (DRN)

Drain workflow:
1. reset previous drain objects
2. build cleaned drain inputs
3. build raster-based stream/wetland drains
4. add optional surface-seepage drains
5. clean overlaps / tiny isolated drain clusters

Important logic in this workflow:
- drains are removed from active lake cells
- drains are removed from GHB cells
- raster drain stage is rebuilt from land surface (`top2d - DRN_DEPTH_M`)


In [ ]:
# Force reset before rebuilding
drn_rec = []
surf_drn_rec = []
print("drn_rec and surf_drn_rec reset to empty")

In [ ]:
# =========================================================
# PREP CLEAN DRAIN INPUTS
# Run AFTER:
#   - lake_mask_2d exists
#   - ghb_cells_df exists
# and BEFORE build_drn_from_raster(...)
# =========================================================

DRN_DEPTH_M = 2.0   # drain stage 2.0 m below land surface

# -----------------------------
# GHB mask on model grid
# -----------------------------
ghb_mask_2d = np.zeros((nrow, ncol), dtype=bool)
for r in ghb_cells_df.itertuples(index=False):
    i, j = int(r.i), int(r.j)
    if (0 <= i < nrow) and (0 <= j < ncol):
        ghb_mask_2d[i, j] = True

# -----------------------------
# Lake mask
# -----------------------------
if "lake_mask_2d" in globals():
    lake_mask = np.asarray(lake_mask_2d, dtype=bool) & (id2d == 1)
else:
    lake_mask = np.zeros((nrow, ncol), dtype=bool)

# -----------------------------
# Start from your current drain rasters
# -----------------------------
drain_frac2d_clean = np.array(drain_frac2d, dtype=float, copy=True)
drain_stage2d = np.full((nrow, ncol), np.nan, dtype=float)

# only where drain fraction exists in active cells
cand = (id2d == 1) & np.isfinite(drain_frac2d_clean) & (drain_frac2d_clean > 0)

# rebuild drain stage from land surface, not from raw drain_elev2d
drain_stage2d[cand] = top2d[cand] - DRN_DEPTH_M

# remove drains from lake cells
drain_frac2d_clean[lake_mask] = 0.0
drain_stage2d[lake_mask] = np.nan

# remove drains from GHB cells
drain_frac2d_clean[ghb_mask_2d] = 0.0
drain_stage2d[ghb_mask_2d] = np.nan

# remove inactive cells
drain_frac2d_clean[id2d <= 0] = 0.0
drain_stage2d[id2d <= 0] = np.nan

# clip area fraction
drain_frac2d_clean = np.clip(drain_frac2d_clean, 0.0, 1.0)

print("Drain candidate cells before exclusions:",
      int(np.count_nonzero((np.asarray(drain_frac2d, dtype=float) > 0) & (id2d == 1))))
print("Lake cells removed from drains:",
      int(np.count_nonzero((np.asarray(drain_frac2d, dtype=float) > 0) & lake_mask)))
print("GHB cells removed from drains:",
      int(np.count_nonzero((np.asarray(drain_frac2d, dtype=float) > 0) & ghb_mask_2d)))
print("Drain cells remaining:",
      int(np.count_nonzero((drain_frac2d_clean > 0) & np.isfinite(drain_stage2d))))

In [ ]:
dfDrn, drn_rec = build_drn_from_raster(
    drain_elev2d=drain_stage2d,
    drain_frac2d=drain_frac2d_clean,
    idomain=idomain,
    top2d=top2d,
    botm3d=botm3d,
    hk3d=hk3d,
    delr=delr,
    delc=delc,
    k_divisor=DRN_K_DIVISOR,
    cond_mult=DRN_COND_MULT,
    min_thick=DRN_MIN_THICK,
    min_area_frac=DRN_MIN_AREA_FRAC,
    elev_eps=DRN_ELEV_EPS,
)


In [ ]:
# =========================================================
# SURFACE SEEPAGE FACES (horizontal surface drains)
# Eq. 13: Csurf = K * Ac / Tsoil
# Specified in every active land cell WITHOUT a river,
# wetland drain, or GHB (lake) cell.
# Cells with zero recharge are excluded to prevent
# draining cells that have no recharge source.
# =========================================================

TSOIL_M          = 50.0    # soil thickness (m) — calibration parameter
SURF_AREA_FRAC   = 0.015    # fraction of cell area for seepage paths
SURF_COND_CAP    = 1e4     # m²/day hard cap on conductance
SURF_ELEV_OFFSET = 2.0    # drain sits 10 cm below land surface
SURF_DRN_LAY     = 0       # always top layer, meaning we are placing the drains on the land surface

# Minimum recharge threshold to qualify for a surface drain
# Cells below this have no recharge source and must not have a drain
# 1e-6 m/day = ~0.37 mm/yr — effectively zero

MIN_RECHARGE_MDAY = 0.0   # below this = weak drain zone
SURF_COND_WEAK    = 0.05   # m²/day — prevents artesian, won't drain aquifer

# ---- save stream/wetland count before merging ----
n_stream_wet = len(drn_rec)

# ---- function to convert existing DRN records to (i, j) cell pairs ----
def rec_to_ij_set(rec_list):
    s = set()
    for rec in rec_list:
        try:
            k, i, j = rec[0]
            s.add((int(i), int(j)))
        except Exception:
            continue
    return s

# ---- function to convert GHB cells DataFrame to (i, j) cell pairs ----
def ghb_to_ij_set(df):
    s = set()
    for r in df.itertuples(index=False):
        s.add((int(r.i), int(r.j)))
    return s

# now that we GHB and drains we exclude those cells from receiving surface seepage drains, along with lake cells
existing_drn_ij = rec_to_ij_set(drn_rec)
existing_ghb_ij = ghb_to_ij_set(ghb_cells_df)

lake_ij = set()
if "lake_mask_2d" in globals():
    li, lj = np.where(np.asarray(lake_mask_2d, dtype=bool))
    lake_ij = set(zip(li.astype(int), lj.astype(int)))

# combine all the excluded cell sets that the cells should not receive surface drains
excluded_ij = existing_drn_ij | existing_ghb_ij | lake_ij

# ---- mean annual recharge across all periods ----
rch_all = np.array([rch_spd[p] for p in range(len(rch_spd))],
                   dtype=float)
rch_mean_annual = np.nanmean(rch_all, axis=0)
rch_mean_annual[~np.isfinite(rch_mean_annual)] = 0.0

print("Active cells:", int(np.sum(idomain[SURF_DRN_LAY] > 0)))
print("Existing raster drain cells:", len(existing_drn_ij))
print("Existing GHB cells:", len(existing_ghb_ij))
print("Lake cells:", len(lake_ij))

# Build the mask of cells that are still eligible for surface seepage drains after excluding existing drains, GHB cells, and lake cells
remaining_mask = np.ones((nrow, ncol), dtype=bool)
remaining_mask &= (idomain[SURF_DRN_LAY] > 0)

excluded_mask = np.zeros((nrow, ncol), dtype=bool)
for i, j in excluded_ij:
    if 0 <= i < nrow and 0 <= j < ncol:
        excluded_mask[i, j] = True
# Remove excluded cells from the remaining mask, that puts recharge values eligble to receive surface drains to zero, and ineligible cells to False
remaining_mask &= (~excluded_mask)

print("Active cells left after exclusions:", int(remaining_mask.sum()))

vals = rch_mean_annual[remaining_mask]
vals = vals[np.isfinite(vals)]
print("Recharge summary in remaining cells:")
if vals.size > 0:
    print("  min   :", vals.min())
    print("  p50   :", np.percentile(vals, 50))
    print("  p95   :", np.percentile(vals, 95))
    print("  max   :", vals.max())
    print("  >= MIN_RECHARGE_MDAY :", int(np.sum(vals >= MIN_RECHARGE_MDAY)))
else:
    print("  no cells remaining")

# ---- build surface seepage records or containers ----
surf_drn_rec = []
surf_rows    = []
n_skipped_inactive = 0
n_skipped_excluded = 0
n_weak_drain       = 0
n_normal_drain     = 0
for i in range(nrow):
    for j in range(ncol):
        if idomain[SURF_DRN_LAY, i, j] <= 0:
            n_skipped_inactive += 1
            continue
        if (i, j) in excluded_ij:
            n_skipped_excluded += 1
            continue

        elev = float(top2d[i, j]) - SURF_ELEV_OFFSET
        K    = float(hk3d[0, i, j])
        Ac   = float(delr[j] * delc[i])

        if rch_mean_annual[i, j] <= 0.0:
            Csurf = SURF_COND_WEAK
            n_weak_drain += 1
        else:
            Csurf_physical = K * (Ac * SURF_AREA_FRAC) / TSOIL_M
            rch_flux  = rch_mean_annual[i, j] * Ac
            # don't force drain to exceeed recharge
            Csurf_min = rch_flux * 2.0 # 1.0x means drain capacity = recharge flux

            Csurf = max(Csurf_physical, Csurf_min)
            Csurf = min(Csurf, SURF_COND_CAP)
            n_normal_drain += 1

        if not np.isfinite(Csurf) or Csurf <= 0:
            continue

        surf_drn_rec.append(((SURF_DRN_LAY, i, j), elev, Csurf))
        surf_rows.append({
            "lay": SURF_DRN_LAY, "row": i, "col": j,
            "elev": elev, "K": K, "Ac": Ac, "cond": Csurf,
        })

dfSurfDrn = pd.DataFrame(surf_rows)
drn_rec = drn_rec + surf_drn_rec

print(f"Stream/wetland DRN cells:          {n_stream_wet:,}")
print(f"Skipped — inactive:                {n_skipped_inactive:,}")
print(f"Skipped — existing drain or GHB:   {n_skipped_excluded:,}")
print(f"Normal surface seepage cells:      {n_normal_drain:,}")
print(f"Weak drain cells (low recharge):   {n_weak_drain:,}")
print(f"Total surface seepage cells:       {len(surf_drn_rec):,}")
if len(dfSurfDrn) > 0:
    print(f"Csurf min/max: "
          f"{dfSurfDrn['cond'].min():.3e} / {dfSurfDrn['cond'].max():.3e}")
print(f"Total DRN records:                 {len(drn_rec):,}")

In [ ]:
from scipy.ndimage import label

# ---------------------------------------------------------
# HELPER FUNCTIONS
# ---------------------------------------------------------
def rec_list_to_bool_mask(rec_list, nrow, ncol):
    mask = np.zeros((nrow, ncol), dtype=bool)
    if rec_list is None:
        return mask
    for rec in rec_list:
        try:
            k, i, j = rec[0]
            mask[int(i), int(j)] = True
        except Exception:
            continue
    return mask

def ghb_df_to_bool_mask(df, nrow, ncol):
    mask = np.zeros((nrow, ncol), dtype=bool)
    if df is None or len(df) == 0:
        return mask
    for r in df.itertuples(index=False):
        mask[int(r.i), int(r.j)] = True
    return mask

# ---------------------------------------------------------
# GUARDS
# ---------------------------------------------------------
if "drn_rec" not in globals():
    raise RuntimeError("drn_rec is not defined. Run the DRN build cell first.")
if "surf_drn_rec" not in globals():
    raise RuntimeError("surf_drn_rec is not defined. Run the surface seepage cell first.")
if "ghb_cells_df" not in globals():
    raise RuntimeError("ghb_cells_df is not defined. Run the GHB cell first.")

# ---------------------------------------------------------
# SPLIT: stream/wetland vs surface seepage
# n_stream_wet was saved in the surface seepage cell
# ---------------------------------------------------------
stream_wet_rec = drn_rec[:n_stream_wet]   # raster drains only
surf_rec       = drn_rec[n_stream_wet:]   # surface seepage only

print("Stream/wetland records before cleanup:", len(stream_wet_rec))
print("Surface seepage records (not cleaned):", len(surf_rec))

# ---------------------------------------------------------
# BUILD MASKS FOR STREAM/WETLAND ONLY
# ---------------------------------------------------------
drn_mask = rec_list_to_bool_mask(stream_wet_rec, nrow, ncol)
ghb_mask = ghb_df_to_bool_mask(ghb_cells_df, nrow, ncol)

if "lake_mask_2d" in globals():
    lake_mask = np.asarray(lake_mask_2d, dtype=bool)
else:
    lake_mask = np.zeros((nrow, ncol), dtype=bool)

# remove GHB and lake overlap from stream/wetland drains
drn_mask = drn_mask & (~ghb_mask) & (~lake_mask)

# ---------------------------------------------------------
# REMOVE TINY ISOLATED COMPONENTS (stream/wetland only)
# ---------------------------------------------------------
structure = np.ones((3, 3), dtype=int)
labels, nlab = label(drn_mask, structure=structure)
sizes = np.bincount(labels.ravel())

MIN_CLUSTER_SIZE = 3
keep_labels = np.where(sizes >= MIN_CLUSTER_SIZE)[0]
keep_labels = keep_labels[keep_labels != 0]
drn_mask_clean = np.isin(labels, keep_labels)

print("Stream/wetland cells before cluster filter:", int(drn_mask.sum()))
print("Stream/wetland cells after  cluster filter:", int(drn_mask_clean.sum()))
print("Removed:", int(drn_mask.sum() - drn_mask_clean.sum()))

# ---------------------------------------------------------
# REBUILD STREAM/WETLAND RECORDS FROM CLEAN MASK
# ---------------------------------------------------------
drn_rec_clean = []
for rec in stream_wet_rec:
    try:
        cellid, elev, cond = rec[:3]
        k, i, j = cellid
        if drn_mask_clean[int(i), int(j)]:
            drn_rec_clean.append(rec)
    except Exception:
        continue

# ---------------------------------------------------------
# FINAL MERGE: cleaned stream/wetland + surface seepage unchanged
# surface seepage covers the whole domain so cluster filter
# is not needed and must not be applied to it
# ---------------------------------------------------------
drn_rec = drn_rec_clean + surf_rec

# ---------------------------------------------------------
# CLEAN dfDrn DATAFRAME (stream/wetland rows only)
# ---------------------------------------------------------
if "dfDrn" in globals() and isinstance(dfDrn, pd.DataFrame) and len(dfDrn) > 0:
    if {"row", "col"}.issubset(dfDrn.columns):
        keep = [drn_mask_clean[int(i), int(j)]
                for i, j in zip(dfDrn["row"], dfDrn["col"])]
        dfDrn = dfDrn.loc[keep].copy()
    elif {"i", "j"}.issubset(dfDrn.columns):
        keep = [drn_mask_clean[int(i), int(j)]
                for i, j in zip(dfDrn["i"], dfDrn["j"])]
        dfDrn = dfDrn.loc[keep].copy()

# ---------------------------------------------------------
# SUMMARY
# ---------------------------------------------------------
print("\nStream/wetland DRN after cleanup:", len(drn_rec_clean))
print("Surface seepage DRN (unchanged): ", len(surf_rec))
print("Total final DRN records:         ", len(drn_rec))
if "dfDrn" in globals():
    print("dfDrn rows after cleanup:        ", len(dfDrn))

In [ ]:
# budget ratio check after building drn_rec = drn_rec + surf_drn_rec
# =========================================================
# DRAIN vs RECHARGE CAPACITY CHECK
# =========================================================
rch0 = np.asarray(rch_spd[0], dtype=float)
active2d = (id2d == 1)

total_rch = float(np.nansum(
    rch0[active2d] * delr[np.where(active2d)[1]] * delc[np.where(active2d)[0]]
))

surf_conds = np.array([r[2] for r in surf_drn_rec])
# capacity if heads were 1 m above drain elevation everywhere
surf_capacity = float(np.sum(surf_conds * 1.0))

ratio = surf_capacity / total_rch if total_rch > 0 else np.inf

print("=" * 55)
print("DRAIN vs RECHARGE CAPACITY CHECK")
print("=" * 55)
print(f"  Total recharge period 0 : {total_rch:.3e} m³/day")
print(f"  Surface drain capacity  : {surf_capacity:.3e} m³/day")
print(f"  Ratio (capacity/rch)    : {ratio:.2f}")
print()
if ratio > 10:
    print("  ❌ FAIL — drains will capture everything, water table stays at surface")
elif ratio > 3:
    print("  ⚠️  WARNING — drains are still aggressive, watch for over-drainage")
else:
    print("  ✅ PASS — drain capacity in reasonable range vs recharge")

In [ ]:
# =========================================================
# CORRECTED PARAMETER SENSITIVITY TUNER
# Uses actual HK field instead of mean_K estimate
# =========================================================
import numpy as np

test_cases = [
    # (TSOIL_M, SURF_AREA_FRAC, SURF_COND_CAP, Csurf_min_mult, label)
    (50.0,  0.005,  1e4, 2.0, "current (ratio=0.20)"),
    (50.0,  0.010,  1e4, 2.0, "option A — 2x area frac"),
    (50.0,  0.015,  1e4, 2.0, "option B — 3x area frac ← try this"),
    (50.0,  0.020,  1e4, 2.0, "option C — 4x area frac"),
    (30.0,  0.015,  1e4, 2.0, "option D — lower TSOIL"),
    (30.0,  0.020,  1e4, 2.0, "option E — lower TSOIL + more area"),
]

# actual recharge
rch0 = np.asarray(rch_spd[0], dtype=float)
active2d = (id2d == 1)
if "lake_mask_2d" in globals():
    active2d = active2d & (~lake_mask_2d)

rows_active, cols_active = np.where(active2d)
total_rch = float(np.nansum(
    rch0[active2d] * delr[cols_active] * delc[rows_active]
))

# get actual per-cell K and area values for active non-lake cells
K_vals  = hk3d[0][active2d].astype(float)
Ac_vals = (delr[cols_active] * delc[rows_active]).astype(float)
rch_vals = rch0[active2d].astype(float)
rch_vals = np.where(np.isfinite(rch_vals), rch_vals, 0.0)

print(f"Total recharge period 0 : {total_rch:.3e} m³/day")
print(f"Active land cells       : {len(K_vals):,}")
print(f"Mean HK layer 1         : {K_vals.mean():.4f} m/day")
print(f"Median HK layer 1       : {np.median(K_vals):.4f} m/day")
print()
print(f"{'Label':<40} {'Ratio':>8}   {'Zone'}")
print("-" * 75)

for tsoil, afrac, cap, cmin_mult, label in test_cases:
    # compute conductance for each cell using actual K
    cond_physical = K_vals * (Ac_vals * afrac) / tsoil
    
    # apply Csurf_min per cell
    rch_flux_cell = rch_vals * Ac_vals
    cond_min = rch_flux_cell * cmin_mult
    
    cond = np.maximum(cond_physical, cond_min)
    cond = np.minimum(cond, cap)
    
    # total capacity assuming 1 m head above drain elevation
    total_capacity = float(np.sum(cond * 1.0))
    ratio = total_capacity / total_rch if total_rch > 0 else np.inf

    if ratio > 10:
        zone = "❌ too strong — dry cells expected"
    elif ratio > 2:
        zone = "⚠️  slightly strong"
    elif ratio >= 0.5:
        zone = "✅ target zone"
    elif ratio >= 0.1:
        zone = "⚠️  slightly weak — artesian risk"
    else:
        zone = "❌ too weak — widespread artesian"

    print(f"{label:<40} {ratio:>8.2f}   {zone}")

In [ ]:
# =========================================================
# DRAIN COVERAGE DIAGNOSTIC
# =========================================================
import numpy as np
import matplotlib.pyplot as plt

active2d = (idomain[0] > 0)
total_active = int(active2d.sum())

# masks
raster_drn_mask = np.zeros((nrow, ncol), dtype=bool)
for rec in drn_rec_clean:
    k, i, j = rec[0]
    raster_drn_mask[i, j] = True

surf_drn_mask = np.zeros((nrow, ncol), dtype=bool)
for rec in surf_rec:
    k, i, j = rec[0]
    surf_drn_mask[i, j] = True

ghb_mask_check = np.zeros((nrow, ncol), dtype=bool)
for r in ghb_cells_df.itertuples(index=False):
    ghb_mask_check[int(r.i), int(r.j)] = True

lake_check = np.asarray(lake_mask_2d, dtype=bool) \
             if "lake_mask_2d" in globals() \
             else np.zeros((nrow, ncol), dtype=bool)

print("=" * 55)
print("DRAIN COVERAGE SUMMARY")
print("=" * 55)
print(f"Total active cells          : {total_active:>10,}")
print(f"Raster stream drain cells   : {raster_drn_mask.sum():>10,}  "
      f"({100*raster_drn_mask.sum()/total_active:.1f}%)")
print(f"Surface seepage cells       : {surf_drn_mask.sum():>10,}  "
      f"({100*surf_drn_mask.sum()/total_active:.1f}%)")
print(f"GHB cells                   : {ghb_mask_check.sum():>10,}  "
      f"({100*ghb_mask_check.sum()/total_active:.1f}%)")
print(f"Lake cells                  : {lake_check.sum():>10,}  "
      f"({100*lake_check.sum()/total_active:.1f}%)")
print()

# check drain elevation vs land surface for raster drains
rr_drn, cc_drn = np.where(raster_drn_mask & active2d)
if len(rr_drn) > 0:
    top_at_drn   = top2d[rr_drn, cc_drn]
    stage_at_drn = np.array([
        rec[1] for rec in drn_rec_clean
        if 0 <= rec[0][1] < nrow and 0 <= rec[0][2] < ncol
    ], dtype=float)

    print("Raster drain stage (elevation) stats:")
    print(f"  mean stage    : {stage_at_drn.mean():.2f} m")
    print(f"  mean top2d    : {top_at_drn.mean():.2f} m")
    print(f"  mean (top-stage): {(top_at_drn.mean() - stage_at_drn.mean()):.2f} m below surface")
    print(f"  stages at surface (diff < 0.1m): "
          f"{int(np.sum((top_at_drn - stage_at_drn) < 0.1)):,}")
    print(f"  stages > 1m below surface: "
          f"{int(np.sum((top_at_drn - stage_at_drn) > 1.0)):,}")
    print(f"  stages > 5m below surface: "
          f"{int(np.sum((top_at_drn - stage_at_drn) > 5.0)):,}")

print()
print("KEY QUESTION: Is 69.7% drain coverage realistic?")
print("Expected for Great Lakes Basin at 1km grid: 20-40%")
if raster_drn_mask.sum() / total_active > 0.5:
    print("⚠️  WARNING — drain coverage > 50% suggests drain_elevation.tif")
    print("   may be covering non-stream cells (wetlands, floodplains, or")
    print("   entire DEM rather than just stream channels)")
    print("   ACTION: check your drain_elevation.tif source data")

In [ ]:
# =========================================================
# CHECK ORIGINAL DRAIN ELEVATION RASTER SOURCE
# =========================================================
import numpy as np
import rasterio as rio
import matplotlib.pyplot as plt

print("=" * 55)
print("ORIGINAL DRAIN ELEVATION RASTER CHECK")
print("=" * 55)

with rio.open(nameInputDrainElev) as src:
    drain_raw = src.read(1).astype(float)
    nd = src.nodata
    print(f"Raster shape      : {src.height} x {src.width}")
    print(f"Nodata value      : {nd}")
    print(f"CRS               : {src.crs}")
    print(f"Resolution        : {abs(src.transform.a):.1f} m")

# count valid (non-nodata, non-zero, finite) cells
if nd is not None:
    valid = np.isfinite(drain_raw) & (drain_raw != nd) & (drain_raw != 0)
else:
    valid = np.isfinite(drain_raw) & (drain_raw != 0)

total_raw = drain_raw.size
n_valid   = int(valid.sum())

print(f"\nTotal raster cells : {total_raw:,}")
print(f"Valid drain cells  : {n_valid:,}  ({100*n_valid/total_raw:.1f}%)")
print(f"Zero/nodata cells  : {total_raw - n_valid:,}")

print(f"\nValid drain elevation stats:")
vals = drain_raw[valid]
print(f"  min  : {vals.min():.2f} m")
print(f"  p5   : {np.percentile(vals, 5):.2f} m")
print(f"  p50  : {np.percentile(vals, 50):.2f} m")
print(f"  p95  : {np.percentile(vals, 95):.2f} m")
print(f"  max  : {vals.max():.2f} m")

# check aligned drain fraction (this is what actually controls coverage)
print("\n--- Aligned drain fraction raster ---")
with rio.open(drain_frac_aligned) as src2:
    frac_raw = src2.read(1).astype(float)
    nd2 = src2.nodata

if nd2 is not None:
    frac_raw[frac_raw == nd2] = 0.0
frac_raw = np.clip(frac_raw, 0.0, 1.0)

n_frac_nonzero = int(np.sum(frac_raw > 0))
n_frac_above_threshold = int(np.sum(frac_raw >= 0.01))  # DRN_MIN_AREA_FRAC

print(f"Cells with frac > 0    : {n_frac_nonzero:,}  "
      f"({100*n_frac_nonzero/frac_raw.size:.1f}%)")
print(f"Cells with frac >= 0.01: {n_frac_above_threshold:,}  "
      f"({100*n_frac_above_threshold/frac_raw.size:.1f}%)")

frac_vals = frac_raw[frac_raw > 0]
print(f"\nDrain fraction stats (non-zero cells):")
print(f"  min  : {frac_vals.min():.4f}")
print(f"  p25  : {np.percentile(frac_vals, 25):.4f}")
print(f"  p50  : {np.percentile(frac_vals, 50):.4f}")
print(f"  p75  : {np.percentile(frac_vals, 75):.4f}")
print(f"  max  : {frac_vals.max():.4f}")

print()
if n_frac_above_threshold / frac_raw.size > 0.5:
    print("⚠️  WARNING — drain fraction raster covers > 50% of grid")
    print("   This suggests drain_elevation.tif was derived from the DEM")
    print("   or a wetland/floodplain mask rather than stream channels only")
    print()
    print("   LIKELY CAUSE: drain_elevation.tif covers the entire")
    print("   floodplain or is a DEM derivative, not NHD stream lines")
    print()
    print("   RECOMMENDATION: your drain raster likely came from a process")
    print("   that filled the entire basin. Check how it was created.")
else:
    print("✅ Drain fraction coverage looks reasonable")
    print("   High model grid coverage is due to 1 km cell size —")
    print("   streams pass through most cells at this resolution")
    print("   Proceed with current setup")

In [ ]:
# =========================================================
# COMPLETE DRAIN BUDGET CHECK
# Measures BOTH raster drains AND surface seepage
# =========================================================
import numpy as np

print("=" * 60)
print("COMPLETE DRAIN BUDGET CHECK (raster + surface seepage)")
print("=" * 60)

# --- recharge ---
rch0 = np.asarray(rch_spd[0], dtype=float)
active2d = (id2d == 1)
rows_a, cols_a = np.where(active2d)
total_rch = float(np.nansum(
    rch0[active2d] * delr[cols_a] * delc[rows_a]
))
print(f"\nTotal recharge period 0 : {total_rch:.3e} m³/day")

# --- raster stream drains ---
raster_conds = np.array([rec[2] for rec in drn_rec_clean], dtype=float)
raster_capacity = float(np.sum(raster_conds * 1.0))  # assume 1m head above drain

# --- surface seepage drains ---
surf_conds = np.array([rec[2] for rec in surf_rec], dtype=float)
surf_capacity = float(np.sum(surf_conds * 1.0))

# --- combined ---
total_capacity = raster_capacity + surf_capacity

print(f"\nDrain type breakdown:")
print(f"  Raster stream drains  : {len(drn_rec_clean):>8,} cells  "
      f"capacity = {raster_capacity:.3e} m³/day")
print(f"  Surface seepage drains: {len(surf_rec):>8,} cells  "
      f"capacity = {surf_capacity:.3e} m³/day")
print(f"  Total drain capacity  :           "
      f"          {total_capacity:.3e} m³/day")

ratio_raster = raster_capacity / total_rch
ratio_surf   = surf_capacity   / total_rch
ratio_total  = total_capacity  / total_rch

print(f"\nRatio breakdown:")
print(f"  Raster drains / recharge : {ratio_raster:.2f}")
print(f"  Surface seepage / recharge: {ratio_surf:.2f}")
print(f"  TOTAL capacity / recharge : {ratio_total:.2f}")

print()
if ratio_total > 10:
    print("  ❌ FAIL — total drain capacity >> recharge")
    print("     Water table will be aggressively drained")
    print("     Expect widespread dry cells and DTW near drain elevation")
elif ratio_total > 3:
    print("  ⚠️  WARNING — total drain capacity > 3x recharge")
    print("     Some over-drainage expected but model may still be usable")
elif ratio_total >= 0.5:
    print("  ✅ PASS — total drain capacity in target range")
else:
    print("  ⚠️  WARNING — drains too weak, artesian conditions likely")

# --- raster drain conductance stats ---
print(f"\nRaster drain conductance stats:")
print(f"  min    : {raster_conds.min():.3e} m²/day")
print(f"  p25    : {np.percentile(raster_conds, 25):.3e} m²/day")
print(f"  median : {np.median(raster_conds):.3e} m²/day")
print(f"  p75    : {np.percentile(raster_conds, 75):.3e} m²/day")
print(f"  max    : {raster_conds.max():.3e} m²/day")
print(f"  mean   : {raster_conds.mean():.3e} m²/day")

# --- what fraction of recharge do drains capture at equilibrium ---
print(f"\nAt equilibrium (heads ~1m above drains):")
print(f"  Raster drains would capture : "
      f"{min(100, ratio_raster*100):.0f}% of recharge")
print(f"  Surface seepage would capture: "
      f"{min(100, ratio_surf*100):.0f}% of recharge")

## 13) Plot model inputs

This section makes a compact 3×3 summary of the current model inputs:
- TOP
- BOTM layer 1
- thickness layer 1
- Kh layer 1
- Kv layer 1
- recharge
- starting head
- DRN cells
- IDOMAIN + GHB cells


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.colors import ListedColormap, BoundaryNorm, LinearSegmentedColormap
from matplotlib.ticker import FixedLocator
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
import matplotlib.font_manager as fm

# =========================================================
# HK QUANTILE COLORMAP — 15-class
# =========================================================
hk_colors_15 = [
    "#08088B",  # near black
    '#0D0887',  # deep blue
    '#2D0594',  # blue-purple
    '#5C01A6',  # violet
    '#7E03A8',  # magenta-purple
    '#9C179E',  # magenta
    '#BD3786',  # pink-magenta
    '#D8576B',  # salmon
    '#E8765C',  # light salmon
    '#ED7953',  # orange
    '#F89441',  # bright orange
    '#FB9F3A',  # light orange
    '#FDCA26',  # gold
    '#F0F921',  # yellow
    '#FCFFA4',  # pale yellow
]

hk_cmap = LinearSegmentedColormap.from_list('hk_15class', hk_colors_15, N=512)

N_CLASSES = 15

def compute_quantile_bounds(arr, n_classes):
    """
    Compute quantile-based class boundaries from a 2D array.
    Only uses finite values > 0.001.
    """
    vals = arr[np.isfinite(arr) & (arr > 0.001)]
    vals = np.sort(vals)
    if vals.size == 0:
        return np.linspace(0, 1, n_classes + 1), vals
    percentiles = np.linspace(0, 100, n_classes + 1)
    bounds = np.percentile(vals, percentiles)
    bounds = np.unique(bounds)
    return bounds, vals


# ---------------------------------------------------------
# helpers
# ---------------------------------------------------------
def get_plot_extent(xorigin, yorigin, delr, delc, nrow, ncol):
    return [xorigin,
            xorigin + np.sum(delr),
            yorigin,
            yorigin + np.sum(delc)]

def rec_list_to_mask(rec_list, nrow, ncol):
    mask = np.full((nrow, ncol), np.nan, dtype=float)
    if rec_list is None or len(rec_list) == 0:
        return mask
    for rec in rec_list:
        try:
            k, i, j = rec[0]
            if 0 <= i < nrow and 0 <= j < ncol:
                mask[i, j] = 1.0
        except Exception:
            continue
    return mask

def ghb_df_to_mask(ghb_cells_df, nrow, ncol):
    mask = np.full((nrow, ncol), np.nan, dtype=float)
    if ghb_cells_df is None or len(ghb_cells_df) == 0:
        return mask
    for r in ghb_cells_df.itertuples(index=False):
        i, j = int(r.i), int(r.j)
        if 0 <= i < nrow and 0 <= j < ncol:
            mask[i, j] = 1.0
    return mask

def active_array(arr, idomain2d):
    out = np.array(arr, dtype=float).copy()
    out[idomain2d <= 0] = np.nan
    return out

def pct(arr, lo, hi):
    v = arr[np.isfinite(arr)]
    return (float(np.nanpercentile(v, lo)),
            float(np.nanpercentile(v, hi))) if v.size else (0, 1)

def add_scalebar(ax, length_km=100):
    fp = fm.FontProperties(size=9)
    ax.add_artist(AnchoredSizeBar(
        ax.transData, length_km * 1000,
        f"{length_km} km", "lower left",
        pad=0.3, color="black", frameon=False,
        size_vertical=max(delc[0], delr[0]) * 1.5,
        fontproperties=fp))

def add_north(ax, x=0.92, y=0.90, size=0.09):
    ax.annotate("N", xy=(x, y), xytext=(x, y - size),
                xycoords="axes fraction", textcoords="axes fraction",
                ha="center", va="center",
                fontsize=11, fontweight="bold",
                arrowprops=dict(facecolor="black", edgecolor="black",
                                width=2.5, headwidth=9))

def add_cbar(fig, ax, im, label, fontsize=9):
    cb = fig.colorbar(im, ax=ax, shrink=0.82, pad=0.02)
    cb.set_label(label, fontsize=fontsize)
    cb.ax.tick_params(labelsize=8)
    return cb

def relabel_log_cbar(cb):
    ticks = cb.get_ticks()
    cb.ax.yaxis.set_major_locator(FixedLocator(ticks))
    cb.set_ticklabels([f"{10**t:.2g}" for t in ticks])

# ---------------------------------------------------------
# prepare domain
# ---------------------------------------------------------
extent = get_plot_extent(xorigin, yorigin, delr, delc, nrow, ncol)

idomain2d = idomain[0] if idomain.ndim == 3 else id2d

# GHB cell mask directly from rebuilt GHB cells
ring_mask_2d = np.zeros((nrow, ncol), dtype=bool)
if "ghb_cells_df" in globals():
    for r in ghb_cells_df.itertuples(index=False):
        i, j = int(r.i), int(r.j)
        if 0 <= i < nrow and 0 <= j < ncol:
            ring_mask_2d[i, j] = True

n_ring = int(ring_mask_2d.sum())
print(f"GHB cells identified for overlay: {n_ring:,}")

def outline_ring(ax):
    if n_ring == 0:
        return
    ax.contour(
        ring_mask_2d.astype(float),
        levels=[0.5],
        colors="#cc0033",
        linewidths=0.6,
        extent=extent,
        origin="upper",
    )

# ---------------------------------------------------------
# arrays in MODEL UNITS as they enter MODFLOW 6
# ---------------------------------------------------------
top_plot = active_array(top2d, idomain2d)
bot1_plot = active_array(botm3d[0], idomain2d)
thk1_plot = active_array(top2d - botm3d[0], idomain2d)

hk1_plot = active_array(hk3d[0], idomain2d)
k33_plot = active_array(k33_3d[0], idomain2d)

rch_all  = np.array([rch_spd[p] for p in range(len(rch_spd))], dtype=float)
rch_mean = np.nanmean(rch_all, axis=0)
rch_mday = active_array(rch_mean, idomain2d)

strt1_plot = active_array(strt[0] if strt.ndim == 3 else strt, idomain2d)

idomain_plot = np.where(idomain2d == 1, 1.0, np.nan)

drn_mask = rec_list_to_mask(drn_rec, nrow, ncol)

ghb_mask = ghb_df_to_mask(ghb_cells_df, nrow, ncol) \
           if "ghb_cells_df" in globals() \
           else np.full((nrow, ncol), np.nan, dtype=float)

# ---------------------------------------------------------
# color limits — computed on terrestrial cells only
# ---------------------------------------------------------
terrestrial_mask = (idomain2d == 1) & (~ring_mask_2d)

def pct_terrestrial(arr, lo, hi):
    v = arr[terrestrial_mask & np.isfinite(arr)]
    return (float(np.nanpercentile(v, lo)),
            float(np.nanpercentile(v, hi))) if v.size else (0, 1)

top_vmin,  top_vmax  = pct_terrestrial(top_plot,  2, 98)
bot_vmin,  bot_vmax  = pct_terrestrial(bot1_plot, 2, 98)
thk_vmin,  thk_vmax  = pct_terrestrial(thk1_plot, 2, 98)

# =========================================================
# QUANTILE BOUNDS for Kh and Kv (terrestrial cells only)
# =========================================================
hk_terr = hk3d[0].copy()
hk_terr[~terrestrial_mask] = np.nan

kv_terr = k33_3d[0].copy()
kv_terr[~terrestrial_mask] = np.nan

hk_bounds, _ = compute_quantile_bounds(hk_terr, N_CLASSES)
kv_bounds, _ = compute_quantile_bounds(kv_terr, N_CLASSES)

hk_norm = BoundaryNorm(hk_bounds, hk_cmap.N)
kv_norm = BoundaryNorm(kv_bounds, hk_cmap.N)

print(f"Kh quantile bounds ({len(hk_bounds)} edges):")
print("  " + ", ".join([f"{b:.4g}" for b in hk_bounds]))
print(f"Kv quantile bounds ({len(kv_bounds)} edges):")
print("  " + ", ".join([f"{b:.4g}" for b in kv_bounds]))

# =========================================================
# Recharge — keep in model units (m/day) for colorbar
# =========================================================
rch_vmin = 0.0
rch_vmax = float(np.nanpercentile(rch_mday[terrestrial_mask], 95))
strt_vmin, strt_vmax = pct_terrestrial(strt1_plot, 2, 98)

# ---------------------------------------------------------
# PLOT — 3x3 grid
# ---------------------------------------------------------
fig, axes = plt.subplots(3, 3, figsize=(19, 17))
axes = axes.ravel()
letters = [f"({x})" for x in "abcdefghi"]

def base_ax(ax, letter, title):
    ax.set_title(f"{letter} {title}", loc="left",
                 fontweight="bold", fontsize=10)
    ax.set_xlabel("Easting (m)", fontsize=8)
    ax.set_ylabel("Northing (m)", fontsize=8)
    ax.tick_params(labelsize=7)

# (a) Top elevation
im = axes[0].imshow(np.ma.masked_invalid(top_plot),
                    origin="upper", extent=extent,
                    cmap="terrain", vmin=top_vmin, vmax=top_vmax,
                    interpolation="nearest")
add_cbar(fig, axes[0], im, "Elevation (m AMSL)")
outline_ring(axes[0])
base_ax(axes[0], letters[0], "Top elevation (m)")
add_north(axes[0]); add_scalebar(axes[0])

# (b) Layer 1 bottom
im = axes[1].imshow(np.ma.masked_invalid(bot1_plot),
                    origin="upper", extent=extent,
                    cmap="terrain", vmin=bot_vmin, vmax=bot_vmax,
                    interpolation="nearest")
add_cbar(fig, axes[1], im, "Elevation (m AMSL)")
outline_ring(axes[1])
base_ax(axes[1], letters[1], "Layer 1 bottom (m)")
add_north(axes[1]); add_scalebar(axes[1])

# (c) Layer 1 thickness
im = axes[2].imshow(np.ma.masked_invalid(thk1_plot),
                    origin="upper", extent=extent,
                    cmap="plasma", vmin=thk_vmin, vmax=thk_vmax,
                    interpolation="nearest")
add_cbar(fig, axes[2], im, "Thickness (m)")
outline_ring(axes[2])
base_ax(axes[2], letters[2], "Layer 1 thickness = TOP − BOTM[0] (m)")
add_north(axes[2]); add_scalebar(axes[2])

# =========================================================
# (d) Kh layer 1 — QUANTILE colormap
# =========================================================
hk1_for_plot = active_array(hk3d[0], idomain2d)

im = axes[3].imshow(np.ma.masked_invalid(hk1_for_plot),
                    origin="upper", extent=extent,
                    cmap=hk_cmap,
                    norm=hk_norm,
                    interpolation="nearest")

cb_hk = fig.colorbar(im, ax=axes[3], shrink=0.82, pad=0.02)
cb_hk.set_label(r"$K_h$ (m/day)", fontsize=9)

# label colorbar ticks at quantile boundaries
tick_positions = hk_bounds
tick_labels = [f"{v:.2g}" for v in hk_bounds]
# show every other tick to avoid crowding
if len(tick_positions) > 8:
    step = max(1, len(tick_positions) // 8)
    tick_positions_show = tick_positions[::step]
    tick_labels_show = tick_labels[::step]
    # always include last
    if tick_positions_show[-1] != tick_positions[-1]:
        tick_positions_show = np.append(tick_positions_show, tick_positions[-1])
        tick_labels_show.append(tick_labels[-1])
else:
    tick_positions_show = tick_positions
    tick_labels_show = tick_labels

cb_hk.set_ticks(tick_positions_show)
cb_hk.set_ticklabels(tick_labels_show)
cb_hk.ax.tick_params(labelsize=7)

outline_ring(axes[3])
base_ax(axes[3], letters[3],
        "Horiz. hydraulic conductivity — quantile (m/day)")
add_north(axes[3]); add_scalebar(axes[3])

# =========================================================
# (e) Kv layer 1 — QUANTILE colormap (same color scale)
# =========================================================
k33_for_plot = active_array(k33_3d[0], idomain2d)

im = axes[4].imshow(np.ma.masked_invalid(k33_for_plot),
                    origin="upper", extent=extent,
                    cmap=hk_cmap,
                    norm=kv_norm,
                    interpolation="nearest")

cb_kv = fig.colorbar(im, ax=axes[4], shrink=0.82, pad=0.02)
cb_kv.set_label(r"$K_v$ (m/day)", fontsize=9)

tick_positions_kv = kv_bounds
tick_labels_kv = [f"{v:.2g}" for v in kv_bounds]
if len(tick_positions_kv) > 8:
    step = max(1, len(tick_positions_kv) // 8)
    tick_positions_kv_show = tick_positions_kv[::step]
    tick_labels_kv_show = tick_labels_kv[::step]
    if tick_positions_kv_show[-1] != tick_positions_kv[-1]:
        tick_positions_kv_show = np.append(tick_positions_kv_show, tick_positions_kv[-1])
        tick_labels_kv_show.append(tick_labels_kv[-1])
else:
    tick_positions_kv_show = tick_positions_kv
    tick_labels_kv_show = tick_labels_kv

cb_kv.set_ticks(tick_positions_kv_show)
cb_kv.set_ticklabels(tick_labels_kv_show)
cb_kv.ax.tick_params(labelsize=7)

outline_ring(axes[4])
base_ax(axes[4], letters[4],
        "Vert. hydraulic conductivity Kh/10 — quantile (m/day)")
add_north(axes[4]); add_scalebar(axes[4])

# =========================================================
# (f) Mean annual recharge — model units (m/day) only
# =========================================================
im = axes[5].imshow(np.ma.masked_invalid(rch_mday),
                    origin="upper", extent=extent,
                    cmap="YlGnBu", vmin=rch_vmin, vmax=rch_vmax,
                    interpolation="nearest")
cb_rch = fig.colorbar(im, ax=axes[5], shrink=0.82, pad=0.02)
cb_rch.set_label("Recharge (m/day)", fontsize=9)
cb_rch.ax.tick_params(labelsize=8)

outline_ring(axes[5])
base_ax(axes[5], letters[5],
        "Mean annual recharge (m/day)")
add_north(axes[5]); add_scalebar(axes[5])

# (g) Starting head
im = axes[6].imshow(np.ma.masked_invalid(strt1_plot),
                    origin="upper", extent=extent,
                    cmap="YlGnBu",
                    vmin=strt_vmin, vmax=strt_vmax,
                    interpolation="nearest")
add_cbar(fig, axes[6], im, "Head (m)")
outline_ring(axes[6])
base_ax(axes[6], letters[6],
        "Starting head (m)")
add_north(axes[6]); add_scalebar(axes[6])

# (h) DRN cells
drn_type = np.zeros((nrow, ncol), dtype=float)
drn_type[idomain2d <= 0] = np.nan
for rec in drn_rec_clean:
    k, i, j = rec[0]
    drn_type[i, j] = 2.0
for rec in surf_rec:
    k, i, j = rec[0]
    if drn_type[i, j] != 2.0:
        drn_type[i, j] = 1.0

cmap_drn = ListedColormap(["#f0f0f0", "#74c2e1", "#1a6b9a"])
axes[7].imshow(np.ma.masked_invalid(drn_type),
               origin="upper", extent=extent,
               cmap=cmap_drn, vmin=0, vmax=2,
               interpolation="nearest")
outline_ring(axes[7])
axes[7].legend(handles=[
    Patch(color="#f0f0f0", label="No drain"),
    Patch(color="#74c2e1",
          label=f"Surface seepage ({len(surf_rec):,} cells)"),
    Patch(color="#1a6b9a",
          label=f"Stream/wetland ({len(drn_rec_clean):,} cells)"),
    Patch(edgecolor="#cc0033", facecolor="none",
          label=f"GHB ring ({n_ring:,} cells)"),
], loc="lower right", fontsize=7, frameon=True)
base_ax(axes[7], letters[7],
        "Drains (streams/wetland vs surface seepage)")
add_north(axes[7]); add_scalebar(axes[7])

# (i) GHB cells — filled ring with terrestrial background
idomain_bg = idomain2d.astype(float)
axes[8].imshow(idomain_bg,
               origin="upper", extent=extent,
               cmap=ListedColormap(["#2b2b2b", "#e8e8e8"]),
               vmin=0, vmax=1,
               interpolation="nearest")

axes[8].imshow(np.ma.masked_invalid(ghb_mask),
               origin="upper", extent=extent,
               cmap=ListedColormap(["#c81d11"]),
               vmin=0.5, vmax=1.5,
               interpolation="nearest")

axes[8].legend(handles=[
    Patch(color="#e8e8e8", label=f"IDOMAIN = 1 (active, {int((idomain2d == 1).sum()):,} cells)"),
    Patch(color="#2b2b2b", label=f"IDOMAIN = 0 (inactive)"),
    Patch(color="#c81d11",
          label=f"GHB cells — {int(np.isfinite(ghb_mask).sum()):,}\n"
                f"(10 km ring inside lakes)"),
], loc="lower right", fontsize=7, frameon=True)
base_ax(axes[8], letters[8],
        "IDOMAIN + GHB cells — Great Lakes boundary")
add_north(axes[8]); add_scalebar(axes[8])

plt.suptitle(
    "Great Lakes Basin MODFLOW 6 — Input Parameters (GHB ring outlined in red)",
    fontsize=13, fontweight="bold", y=1.005)
plt.tight_layout()
plt.savefig(r"D:\Users\abolmaal\modelling\Figs\Testing_6\modelinputs.jpeg",
            dpi=300, bbox_inches="tight")
plt.show()

# ---------------------------------------------------------
# summary in model units, split by terrestrial vs ring
# ---------------------------------------------------------
terr = terrestrial_mask
ring = ring_mask_2d

print("=== MODEL INPUT SUMMARY ===")
print(f"Active cells total:        {int(np.sum(idomain2d == 1)):>10,}")
print(f"  terrestrial (land):      {int(terr.sum()):>10,}")
print(f"  GHB ring (in lake):      {int(ring.sum()):>10,}")
print()
print("                       terrestrial        |   GHB ring")
print("                  min      med     max    |  min    med    max")
def row(name, arr, unit=""):
    t = arr[terr][np.isfinite(arr[terr])]
    r = arr[ring][np.isfinite(arr[ring])]
    print(f"{name:8s} "
          f"{t.min():>7.2g} {np.median(t):>7.2g} {t.max():>7.2g} "
          f"| {r.min():>6.2g} {np.median(r):>6.2g} {r.max():>6.2g} {unit}")
row("TOP",   top2d,                   "m")
row("BOTM0", botm3d[0],               "m")
row("THK1",  top2d - botm3d[0],       "m")
row("HK0",   hk3d[0],                 "m/day")
row("K33",   k33_3d[0],               "m/day")
row("STRT",  strt[0] if strt.ndim == 3 else strt, "m")
row("RCH",   rch_mean,                "m/day")
print()
print(f"DRN cells:  {len(drn_rec):,}")
print(f"GHB cells:  {int(np.isfinite(ghb_mask).sum()):,}")

## 14) Pre-run sanity checks

This is a good checkpoint before writing and running MODFLOW 6.
It checks:
- DRN conductance ranges
- duplicate drains
- DRN / GHB overlap
- GHB stage consistency
- recharge and head bounds


In [ ]:
print("Recharge totals by stress period:")
for p in sorted(rch_spd.keys()):
    arr = np.asarray(rch_spd[p], dtype=float)
    arr = np.where(np.isfinite(arr), arr, 0.0)

    total = arr.sum() * float(delr[0] * delc[0])   # approximate if uniform 1000 m grid
    nonzero = int(np.count_nonzero(arr > 0))
    vmax = float(arr.max()) if arr.size else np.nan

    print(f"  period {p:2d}: total={total:.3e} m3/day, nonzero cells={nonzero:,}, max={vmax:.3e}")

In [ ]:
terr_mask = (idomain[0] > 0)
if "lake_mask_2d" in globals():
    terr_mask = terr_mask & (~lake_mask_2d)

print("\nRecharge over active land cells by period:")
for p in sorted(rch_spd.keys()):
    arr = np.asarray(rch_spd[p], dtype=float)
    vals = arr[terr_mask]
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        print(f"  period {p:2d}: no values")
    else:
        print(f"  period {p:2d}: min={vals.min():.3e}, p50={np.percentile(vals,50):.3e}, "
              f"p95={np.percentile(vals,95):.3e}, max={vals.max():.3e}, "
              f"nonzero={np.count_nonzero(vals>0):,}")

In [ ]:
import numpy as np
import pandas as pd

print("=" * 60)
print("SANITY CHECKS BEFORE RUNNING MODFLOW 6")
print("=" * 60)

# ---------------------------------------------------------
# 1) SURFACE SEEPAGE CONDUCTANCE
# ---------------------------------------------------------
print("\n--- 1) Surface seepage conductance ---")
surf_conds = np.array([r[2] for r in surf_drn_rec])
print(f"  Csurf min:    {surf_conds.min():.3e} m²/day")
print(f"  Csurf max:    {surf_conds.max():.3e} m²/day")
print(f"  Csurf median: {np.median(surf_conds):.3e} m²/day")
print(f"  Cells at cap (1e4): {np.sum(surf_conds >= 9999):.0f}")

if surf_conds.max() > 1e6:
    print("  ❌ FAIL — Csurf max > 1e6, TSOIL/SURF_AREA_FRAC fix not applied")
elif surf_conds.max() > 1e5:
    print("  ⚠️  WARNING — Csurf max > 1e5, consider lowering SURF_COND_CAP")
else:
    print("  ✅ PASS — conductance range looks reasonable")

# ---------------------------------------------------------
# 2) DRAIN RECORD COUNTS AND NO DUPLICATES
# ---------------------------------------------------------
print("\n--- 2) DRN record counts ---")
print(f"  Stream/wetland records: {len(drn_rec_clean):,}")
print(f"  Surface seepage records: {len(surf_rec):,}")
print(f"  Total drn_rec: {len(drn_rec):,}")

# check for duplicate (i,j) cells in final drn_rec
ij_list = [(r[0][1], r[0][2]) for r in drn_rec]
ij_set  = set(ij_list)
n_dups  = len(ij_list) - len(ij_set)
print(f"  Duplicate (i,j) cells: {n_dups}")
if n_dups > 0:
    print("  ❌ FAIL — duplicate drain cells will cause MODFLOW budget errors")
else:
    print("  ✅ PASS — no duplicate drain cells")

# ---------------------------------------------------------
# 3) OVERLAP BETWEEN DRN AND GHB
# ---------------------------------------------------------
print("\n--- 3) DRN / GHB overlap ---")
drn_ij = {(r[0][1], r[0][2]) for r in drn_rec}
ghb_ij = {(int(r.i), int(r.j)) for r in ghb_cells_df.itertuples()}
overlap = drn_ij & ghb_ij
print(f"  DRN cells:  {len(drn_ij):,}")
print(f"  GHB cells:  {len(ghb_ij):,}")
print(f"  Overlap:    {len(overlap)}")
if len(overlap) > 0:
    print("  ❌ FAIL — DRN and GHB share cells, remove overlap before running")
else:
    print("  ✅ PASS — no DRN/GHB overlap")

# ---------------------------------------------------------
# 4) GHB STAGE vs CELL TOP
# ---------------------------------------------------------
print("\n--- 4) GHB stage vs land surface (top_cap check) ---")
if "top_cap" not in ghb_cells_df.columns:
    print("  ⚠️  top_cap column missing — GHB stage cap fix not applied")
else:
    rr = ghb_cells_df["i"].to_numpy(dtype=int)
    cc = ghb_cells_df["j"].to_numpy(dtype=int)
    first_per_stages = np.array([
        float(monthly_stages_model.loc[model_months[0], r.stage_name])
        for r in ghb_cells_df.itertuples(index=False)
    ])
    capped_stages = np.array([
        min(float(monthly_stages_model.loc[model_months[0], r.stage_name]),
            float(r.top_cap) - 0.10)
        for r in ghb_cells_df.itertuples(index=False)
    ])
    n_above = np.sum(first_per_stages > top2d[rr, cc])
    n_capped = np.sum(capped_stages < first_per_stages)
    print(f"  GHB cells where raw stage > top2d:   {n_above:,}")
    print(f"  GHB cells where stage was capped:    {n_capped:,}")
    if n_above > 0 and n_capped == 0:
        print("  ❌ FAIL — stage cap fix not active in ghb_spd loop")
    else:
        print("  ✅ PASS — stage cap applied to all cells above land surface")

# ---------------------------------------------------------
# 5) STARTING HEADS WITHIN VALID RANGE
# ---------------------------------------------------------
print("\n--- 5) Starting heads ---")
active = idomain[0] > 0
strt1 = strt[0]
above_top  = np.sum(strt1[active] > top2d[active])
below_bot  = np.sum(strt1[active] < botm3d[0][active])
print(f"  Cells with head > top2d:   {above_top:,}")
print(f"  Cells with head < botm[0]: {below_bot:,}")
print(f"  Head min/max (active): "
      f"{strt1[active].min():.1f} / {strt1[active].max():.1f} m")
if above_top > 0:
    print("  ⚠️  WARNING — some starting heads above land surface")
if below_bot > 0:
    print("  ❌ FAIL — some starting heads below cell bottom, model will fail")
else:
    print("  ✅ PASS — all starting heads above cell bottom")

# ---------------------------------------------------------
# 6) LAYER THICKNESS
# ---------------------------------------------------------
print("\n--- 6) Layer thickness ---")
for k in range(nlay):
    if k == 0:
        thk = top2d - botm3d[0]
    else:
        thk = botm3d[k-1] - botm3d[k]
    thk_active = thk[idomain[k] > 0]
    n_thin = np.sum(thk_active <= 0)
    print(f"  Layer {k+1}: min={thk_active.min():.2f} m  "
          f"max={thk_active.max():.1f} m  "
          f"cells <= 0m: {n_thin}")
    if n_thin > 0:
        print(f"  ❌ FAIL — Layer {k+1} has zero or negative thickness cells")

# ---------------------------------------------------------
# 7) RECHARGE BUDGET VS DRAIN CAPACITY
# ---------------------------------------------------------
print("\n--- 7) Recharge vs drain budget (period 0) ---")
rch0 = rch_spd[0]
active2d = (id2d == 1)

total_rch_m3day = float(np.sum(
    rch0[active2d] * delr[np.where(active2d)[1]] * delc[np.where(active2d)[0]]
))

# surface seepage max possible outflow (if all heads at drain elev)
surf_cond_arr = np.array([r[2] for r in surf_drn_rec])
# max outflow if head were 1 m above drain elevation everywhere
surf_max_out = float(np.sum(surf_cond_arr * 1.0))

print(f"  Total recharge period 0:        {total_rch_m3day:.3e} m³/day")
print(f"  Surface seepage max outflow*:   {surf_max_out:.3e} m³/day")
print(f"  (* if heads were 1m above drain elevation everywhere)")
ratio = surf_max_out / total_rch_m3day if total_rch_m3day > 0 else np.inf
print(f"  Ratio (seepage capacity / recharge): {ratio:.2f}")
if ratio > 100:
    print("  ❌ FAIL — drain capacity >> recharge, model will over-drain")
elif ratio > 10:
    print("  ⚠️  WARNING — drain capacity 10x recharge, watch for drainage")
else:
    print("  ✅ PASS — drain capacity in reasonable range vs recharge")

# ---------------------------------------------------------
# 8) HK RANGE
# ---------------------------------------------------------
print("\n--- 8) Hydraulic conductivity ---")
for k in range(nlay):
    hk_active = hk3d[k][idomain[k] > 0]
    print(f"  Layer {k+1}: min={hk_active.min():.2e}  "
          f"max={hk_active.max():.2e}  "
          f"median={np.median(hk_active):.2e} m/day")

# ---------------------------------------------------------
# 9) STRESS PERIOD COUNT CONSISTENCY
# ---------------------------------------------------------
print("\n--- 9) Stress period consistency ---")
print(f"  perioddata_run:  {len(perioddata_run)} periods")
print(f"  rch_spd:         {len(rch_spd)} periods")
print(f"  ghb_spd:         {len(ghb_spd)} periods")

ok = (len(rch_spd) == len(perioddata_run) == len(ghb_spd))
if not ok:
    print("  ❌ FAIL — period counts do not match")
else:
    print("  ✅ PASS — all packages have same number of stress periods")

# ---------------------------------------------------------
# 10) k33 ANISOTROPY
# ---------------------------------------------------------
print("\n--- 10) Vertical K (k33) anisotropy ---")
if "k33_3d" not in globals():
    print("  ❌ FAIL — k33_3d not defined, NPF will use isotropic K")
else:
    for k in range(nlay):
        hk_act = hk3d[k][idomain[k] > 0]
        kv_act = k33_3d[k][idomain[k] > 0]
        ratio  = hk_act / kv_act
        print(f"  Layer {k+1}: Kh/Kv ratio "
              f"min={ratio.min():.1f}  max={ratio.max():.1f}  "
              f"median={np.median(ratio):.1f}")
    print("  ✅ k33_3d defined")

print("\n" + "=" * 60)
print("SANITY CHECKS COMPLETE")
print("=" * 60)

In [ ]:
active = (idomain > 0)
bad_k = active & (~np.isfinite(hk3d) | (hk3d <= 0))

print("Active cells with bad K:", int(bad_k.sum()))
if bad_k.any():
    print("First 20 bad cells:")
    print(np.argwhere(bad_k)[:20])

## 15) Build and run MODFLOW 6

This section writes the simulation and runs it.


In [ ]:
# run this section after loading the warm-start heads to check their range and make sure they are above the cell bottoms
warmstart_path = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\warmstart_heads.npy"
strt = np.load(warmstart_path).astype("float32")

print("Loaded warm-start heads:", strt.shape)
print("Layer 1 warm-start min/max:",
      strt[0][idomain[0] > 0].min(),
      strt[0][idomain[0] > 0].max())

In [ ]:
# Close possible open handles from earlier tests
for varname in ["cbc", "hds", "bud", "headfile", "cellbudgetfile", "hdobj", "cbbobj"]:
    obj = globals().get(varname, None)
    try:
        if obj is not None and hasattr(obj, "close"):
            obj.close()
            print(f"Closed {varname}")
    except Exception as e:
        print(f"Could not close {varname}: {repr(e)}")

if os.path.isdir(sim_ws):
    safe_rmtree(sim_ws)
os.makedirs(sim_ws, exist_ok=True)

sim = flopy.mf6.MFSimulation(sim_name=nameSim, sim_ws=sim_ws, exe_name=exe_path)

tdis = flopy.mf6.ModflowTdis(
    sim,
    time_units="DAYS",
    nper=len(perioddata_run),
    perioddata=perioddata_run,
    start_date_time=START_DATE,
)
# Steady state spin-up in period 0, then transient periods 1+ with same length as perioddata_run
ims = flopy.mf6.ModflowIms(
    sim,
    pname="ims",
    complexity="MODERATE",
    outer_maximum=500,          # increased from 100
    inner_maximum=300,          # increased from 200
    outer_dvclose=1e-3,         # tightened from 1e-2
    inner_dvclose=1e-3,         # tightened from 1e-2
    rcloserecord=1e-3,          # tightened from 1e-2
    linear_acceleration="BICGSTAB",
    under_relaxation="DBD",
    under_relaxation_theta=0.9,  # add damping
    under_relaxation_kappa=0.0001,
    under_relaxation_gamma=0.0,
    print_option="SUMMARY",
    filename=f"{nameModel}.ims",
)

gwf = flopy.mf6.ModflowGwf(sim, modelname=nameModel, save_flows=True)
sim.register_ims_package(ims, [gwf.name])

dis = flopy.mf6.ModflowGwfdis(
    gwf,
    nlay=nlay, nrow=nrow, ncol=ncol,
    delr=delr, delc=delc,
    top=top2d,
    botm=botm3d,
    idomain=idomain,
    xorigin=xorigin, yorigin=yorigin,
)

icelltype = [1] + [0] * (nlay - 1)

npf = flopy.mf6.ModflowGwfnpf(
    gwf,
    icelltype=icelltype,
    k=hk3d,
    k33=k33_3d,
    save_specific_discharge=True,
)

ic = flopy.mf6.ModflowGwfic(gwf, strt=strt)

# ---- CHANGED: period 0 = steady-state spin-up, periods 1+ = transient ----
sto = flopy.mf6.ModflowGwfsto(
    gwf,
    ss=1e-6,
    sy=0.1,
    steady_state=False,
    transient=True,
)


oc = flopy.mf6.ModflowGwfoc(
    gwf,
    head_filerecord=f"{nameModel}.hds",
    budget_filerecord=f"{nameModel}.cbb",
    saverecord=[("HEAD", "ALL"), ("BUDGET", "ALL")],
    printrecord=[("HEAD", "LAST"), ("BUDGET", "LAST")],
)

drn_rec = drn_rec if "drn_rec" in globals() else []
wet_drn_rec = wet_drn_rec if "wet_drn_rec" in globals() else []

if USE_DRN and len(drn_rec) > 0:
    drn_spd = {per: drn_rec for per in range(len(perioddata_run))}
    drn = flopy.mf6.ModflowGwfdrn(
        gwf,
        pname="DRN",
        filename=f"{nameModel}.drn",
        maxbound=len(drn_rec),
        stress_period_data=drn_spd,
        save_flows=True,
    )
    print("✅ DRN package added:", len(drn_rec), "cells")
    print("   DRN stress periods:", len(drn_spd))
else:
    print("Skipping DRN package")

if USE_GHB:
    if "ghb_spd" not in globals():
        raise ValueError("ghb_spd does not exist. Run Part 10 first.")
    if ghb_spd is None:
        raise ValueError("ghb_spd is None. Check Part 10.")
    if len(ghb_spd) == 0:
        raise ValueError("ghb_spd is empty. Check Part 10.")

if USE_GHB and ghb_spd is not None:
    print("Using GHB stress periods:", len(ghb_spd))
    print("Using GHB maxbound:", max(len(v) for v in ghb_spd.values()))
    if "ghb_cells_df" in globals():
        print("Using GHB lakes:", sorted(ghb_cells_df["lake_name"].unique()))

    ghb = flopy.mf6.ModflowGwfghb(
        gwf,
        pname="GHB_gl",
        filename=f"{nameModel}.ghb",
        boundnames=True,
        print_input=True,
        save_flows=True,
        maxbound=max(len(v) for v in ghb_spd.values()),
        stress_period_data=ghb_spd,
    )
    print("✅ GHB package added")
else:
    print("Skipping GHB package")

rcha = flopy.mf6.ModflowGwfrcha(
    gwf,
    pname="RCHA",
    filename=f"{nameModel}.rcha",
    recharge=rch_spd,
)
print("✅ RCH package added:", len(rch_spd), "periods")

# ---- ADDED: final consistency check before writing ----
print("\n--- Final period count check ---")
print(f"perioddata_run: {len(perioddata_run)}")
print(f"rch_spd:        {len(rch_spd)}")
print(f"ghb_spd:        {len(ghb_spd)}")
print(f"drn_spd:        {len(drn_spd)}")
assert len(rch_spd) == len(perioddata_run), \
    f"rch_spd has {len(rch_spd)} periods, expected {len(perioddata_run)}"
assert len(ghb_spd) == len(perioddata_run), \
    f"ghb_spd has {len(ghb_spd)} periods, expected {len(perioddata_run)}"
assert len(drn_spd) == len(perioddata_run), \
    f"drn_spd has {len(drn_spd)} periods, expected {len(perioddata_run)}"
print("✅ All period counts match — safe to write and run")

def rec_to_set(rec_list):
    out = set()
    for rec in rec_list:
        try:
            k, i, j = rec[0]
            out.add((int(k), int(i), int(j)))
        except Exception:
            pass
    return out

drn_set = rec_to_set(drn_rec)
wet_drn_set = rec_to_set(wet_drn_rec)

ghb_set = set()
if "ghb_spd" in globals() and ghb_spd is not None:
    for per, recs in ghb_spd.items():
        for rec in recs:
            try:
                k, i, j = rec[0]
                ghb_set.add((int(k), int(i), int(j)))
            except Exception:
                pass

print("DRN ∩ wetland DRN:", len(drn_set & wet_drn_set))
print("DRN ∩ GHB:", len(drn_set & ghb_set))
print("wetland DRN ∩ GHB:", len(wet_drn_set & ghb_set))

In [ ]:
sim.write_simulation()
success, buff = sim.run_simulation()
print("Run success:", success)
if not success:
    with open(os.path.join(sim_ws, f"{nameModel}.lst")) as f:
        lines = f.readlines()
    print("".join(lines[-50:]))

## 16) Optional: export warm-start heads from the previous run

Use this if you want to rerun with a better initial head field.


In [ ]:
#Extract the head from this simulation
import flopy.utils.binaryfile as bf
import numpy as np
head_path = os.path.join(sim_ws, f"{nameModel}.hds")
hds = bf.HeadFile(head_path)
times = hds.get_times()
h_final = hds.get_data(totim=times[-1])

h_warmstart = np.array(h_final, dtype="float32", copy=True)

# replace dry / absurd placeholder values
for k in range(nlay):
    bad = (~np.isfinite(h_warmstart[k])) | (np.abs(h_warmstart[k]) >= 1e20)
    h_warmstart[k][bad] = botm3d[k][bad] + 1.0

# inactive cells -> 0
h_warmstart[idomain <= 0] = 0.0

# optional extra guard: keep heads above cell bottom
for k in range(nlay):
    active = idomain[k] > 0
    too_low = active & (h_warmstart[k] <= botm3d[k])
    h_warmstart[k][too_low] = botm3d[k][too_low] + 0.1
    print(f"Layer {k+1}: raised {too_low.sum():,} cells below bottom")

warmstart_path = os.path.join(
    r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers",
    "warmstart_heads.npy"
)
np.save(warmstart_path, h_warmstart)

print(f"\nWarm start heads saved: {warmstart_path}")
print(f"Shape: {h_warmstart.shape}")
print(f"Layer 1 min/max: "
      f"{h_warmstart[0][idomain[0] > 0].min():.1f} / "
      f"{h_warmstart[0][idomain[0] > 0].max():.1f} m")

## 17) Optional: listing-file budget summary

This reads the MF6 listing file and prints the flux budget and percent discrepancy.


In [ ]:
import flopy
import pandas as pd
import numpy as np
import os

# ---------------------------------------------------------
# MODFLOW 6 budget from listing file
# ---------------------------------------------------------
lst_path = os.path.join(sim_ws, f"{nameModel}.lst")

# MF6 uses Mf6ListBudget, not MfListBudget
mf_list = flopy.utils.Mf6ListBudget(lst_path)
df_flux, df_vol = mf_list.get_dataframes(start_datetime=START_DATE, diff=True)

print("=== FLUX BUDGET (m³/day) ===")
print(df_flux.to_string())

print("\n=== PERCENT DISCREPANCY ===")
# discrepancy column is usually named 'PERCENT_DISCREPANCY' or similar
disc_cols = [c for c in df_flux.columns if "DISCREPANCY" in c.upper() or "PERCENT" in c.upper()]
if disc_cols:
    print(df_flux[disc_cols])
else:
    print("Discrepancy column not found — printing all columns:")
    print(df_flux.columns.tolist())

## 18) Optional: example post-processing

The original notebook had many exploratory post-processing cells.  
This notebook keeps one representative post-processing figure cell as a starting point.


In [ ]:
# =========================================================
# DTW DIAGNOSTIC — find where extreme values come from
# =========================================================
import numpy as np

print("=" * 55)
print("DTW DISTRIBUTION DIAGNOSTIC")
print("=" * 55)

# --- overall stats ---
finite_dtw = avg_dtw[np.isfinite(avg_dtw)]
print(f"\nTotal finite DTW cells : {len(finite_dtw):,}")
print(f"  min   : {finite_dtw.min():.2f} m")
print(f"  p1    : {np.percentile(finite_dtw, 1):.2f} m")
print(f"  p5    : {np.percentile(finite_dtw, 5):.2f} m")
print(f"  p25   : {np.percentile(finite_dtw, 25):.2f} m")
print(f"  p50   : {np.percentile(finite_dtw, 50):.2f} m")
print(f"  p75   : {np.percentile(finite_dtw, 75):.2f} m")
print(f"  p95   : {np.percentile(finite_dtw, 95):.2f} m")
print(f"  p99   : {np.percentile(finite_dtw, 99):.2f} m")
print(f"  max   : {finite_dtw.max():.2f} m")

# --- negative values (head above land surface) ---
n_neg = int(np.sum(avg_dtw < 0))
print(f"\nNegative DTW cells (head > land): {n_neg:,}")
if n_neg > 0:
    neg_vals = avg_dtw[avg_dtw < 0]
    print(f"  most negative : {neg_vals.min():.2f} m")
    print(f"  median neg    : {np.median(neg_vals):.2f} m")

# --- extreme positive values (likely dry cells) ---
n_extreme = int(np.sum(avg_dtw > 100))
n_very_extreme = int(np.sum(avg_dtw > 200))
print(f"\nDTW > 100 m (likely dry cells) : {n_extreme:,}")
print(f"DTW > 200 m (almost certainly dry): {n_very_extreme:,}")

# --- check if these are lake cells ---
if "lake_mask_2d" in globals():
    n_extreme_lake = int(np.sum((avg_dtw > 100) & lake_mask_2d))
    n_extreme_land = int(np.sum((avg_dtw > 100) & (~lake_mask_2d) & (idomain[0] > 0)))
    print(f"  of which in lake cells : {n_extreme_lake:,}")
    print(f"  of which in land cells : {n_extreme_land:,}")

# --- check raw heads for dry cell signatures ---
print("\n--- Checking raw head array for dry cell placeholders ---")
for i, kp in enumerate(kstpkper_list[:3]):   # check first 3 periods
    h = hdobj.get_data(kstpkper=kp)
    h0 = h[0]
    n_huge   = int(np.sum(np.abs(h0) >= 1e20))
    n_normal = int(np.sum((np.abs(h0) < 1e20) & (idomain[0] > 0)))
    print(f"  Period {i}: huge values (|h|>=1e20): {n_huge:,}  |  normal active: {n_normal:,}")

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
from matplotlib.colors import TwoSlopeNorm
import flopy
import flopy.utils.binaryfile as bf

# ============================================================
# SETTINGS
# ============================================================
boundary_shp = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Ibound\extended_Bdry_final_GLB_Albers_exported.shp"
save_fig = True
out_fig = r"D:\Users\abolmaal\modelling\Figs\Testing_6\avg_dtw_avg_recharge_studyperiod.png"

# if head is not already loaded
if "head" not in globals():
    head_path = os.path.join(sim_ws, f"{nameModel}.hds")
    hds = bf.HeadFile(head_path)
    times = hds.get_times()
    head = np.array([hds.get_data(totim=t) for t in times], dtype=float)
    print("Loaded head array:", head.shape)

# top from model
top = np.array(gwf.dis.top.array, dtype=float)

# ============================================================
# HELPERS
# ============================================================
def get_water_table(head3d, idomain3d, huge=1e29):
    """
    Water table = first valid head from the top active layer downward.
    Uses huge=1e29 to catch MODFLOW 6 dry-cell placeholder (1e30).
    """
    nlay, nrow, ncol = head3d.shape
    wt = np.full((nrow, ncol), np.nan, dtype=float)

    for k in range(nlay):
        hk = np.array(head3d[k], dtype=float)

        # mask MODFLOW dry-cell placeholder AND inactive cells
        hk[np.abs(hk) >= huge] = np.nan
        hk[idomain3d[k] <= 0] = np.nan

        # also mask any remaining unrealistic values
        hk[hk > 10000]  = np.nan   # no cell top should exceed 10,000 m
        hk[hk < -1000]  = np.nan   # no head below -1000 m

        take = np.isnan(wt) & np.isfinite(hk)
        wt[take] = hk[take]

    return wt

def get_depth_to_water(head_t, idomain, top, clip_negative=True):
    wt = get_water_table(head_t, idomain)
    dtw = np.array(top, dtype=float) - wt
    dtw[~np.isfinite(wt)] = np.nan

    if clip_negative:
        dtw = np.where(np.isfinite(dtw), np.maximum(dtw, 0.0), np.nan)

    return dtw

def robust_limits(arr, qlow=2, qhigh=98, symmetric=False):
    a = np.asarray(arr, dtype=float)
    a = a[np.isfinite(a)]

    if a.size == 0:
        return 0.0, 1.0

    vmin = float(np.nanpercentile(a, qlow))
    vmax = float(np.nanpercentile(a, qhigh))

    if symmetric:
        vmax_abs = max(abs(vmin), abs(vmax))
        return -vmax_abs, vmax_abs

    return vmin, vmax

def get_extent(xorigin, yorigin, delr, delc, nrow, ncol):
    if np.ndim(delr) == 0:
        width = ncol * float(delr)
    else:
        width = float(np.sum(delr))

    if np.ndim(delc) == 0:
        height = nrow * float(delc)
    else:
        height = float(np.sum(delc))

    return [xorigin, xorigin + width, yorigin, yorigin + height]

# ============================================================
# 1) AVERAGE DEPTH TO WATER
# ============================================================
# =========================================================
# COMPUTE DTW — with dry cell protection
# =========================================================

dtw_all = []
for i in range(head.shape[0]):
    wt = get_water_table(head[i], idomain, huge=1e29)
    dtw = np.array(top, dtype=float) - wt

    # hard clip: anything beyond physical bounds is a dry cell artifact
    # Great Lakes Basin: no cell should have DTW > total aquifer thickness (~500 m max)
    dtw = np.where(dtw > 200, np.nan, dtw)   # treat >200 m as dry/artifact
    dtw = np.where(idomain[0] <= 0, np.nan, dtw)

    dtw_all.append(dtw)

dtw_all = np.array(dtw_all, dtype=float)

# average — nanmean ignores dry-cell NaNs
avg_dtw = np.nanmean(dtw_all, axis=0)

# mask inactive cells
avg_dtw = np.where(idomain[0] > 0, avg_dtw, np.nan)

# mask lake cells
if "lake_mask_2d" in globals():
    avg_dtw = np.where(lake_mask_2d, np.nan, avg_dtw)

# ---- final stats ----
finite = avg_dtw[np.isfinite(avg_dtw)]
print("Average DTW after cleaning:")
print(f"  min  : {np.nanmin(finite):.2f} m")
print(f"  p5   : {np.percentile(finite, 5):.2f} m")
print(f"  p50  : {np.percentile(finite, 50):.2f} m")
print(f"  p95  : {np.percentile(finite, 95):.2f} m")
print(f"  max  : {np.nanmax(finite):.2f} m")
print(f"  negative cells : {int(np.sum(finite < 0))}")
print(f"  cells > 50 m   : {int(np.sum(finite > 50)):,}")
print(f"  cells > 100 m  : {int(np.sum(finite > 100)):,}")
# ============================================================
# 2) AVERAGE RECHARGE
# ============================================================
if isinstance(rch_spd, dict):
    rch_arrays = []
    for per in sorted(rch_spd.keys()):
        arr = np.array(rch_spd[per], dtype=float)

        # if recharge is given as 3D, use top layer / first slice
        if arr.ndim == 3:
            arr = arr[0]

        rch_arrays.append(arr)

    rch_all = np.array(rch_arrays, dtype=float)

else:
    arr = np.array(rch_spd, dtype=float)
    if arr.ndim == 3:
        rch_all = arr
    elif arr.ndim == 2:
        rch_all = arr[np.newaxis, :, :]
    else:
        raise ValueError("rch_spd has an unsupported shape")

avg_rch = np.nanmean(rch_all, axis=0)
avg_rch = np.where(idomain[0] > 0, avg_rch, np.nan)

# ============================================================
# 3) COLOR LIMITS
# ============================================================
dtw_vmin, dtw_vmax = robust_limits(avg_dtw, qlow=2, qhigh=98, symmetric=True)
dtw_vmin = -10.0    # artesian / head above land
dtw_vmax =  50.0    # max realistic DTW for Great Lakes Basin
dtw_norm = TwoSlopeNorm(vmin=dtw_vmin, vcenter=0.0, vmax=dtw_vmax)

rch_vmin, rch_vmax = robust_limits(avg_rch, qlow=2, qhigh=98, symmetric=False)

# ============================================================
# 4) BOUNDARY
# ============================================================
gdf_bdry = gpd.read_file(boundary_shp)
try:
    if getattr(gwf.modelgrid, "crs", None) is not None:
        gdf_bdry = gdf_bdry.to_crs(gwf.modelgrid.crs)
except Exception:
    pass

extent = get_extent(xorigin, yorigin, delr, delc, nrow, ncol)

# ============================================================
# 5) PLOT
# ============================================================
# ============================================================
# 5) PLOT
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)

# ---- Average DTW
ax = axes[0]
im1 = ax.imshow(
    np.ma.masked_invalid(avg_dtw),
    origin="upper",
    extent=extent,
    cmap="RdBu",
    norm=dtw_norm,
)
gdf_bdry.boundary.plot(ax=ax, color="black", linewidth=0.8)
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.text(
    0.02, 0.98, "a)",
    transform=ax.transAxes,
    ha="left", va="top",
    fontsize=14, fontweight="bold"
)
cbar1 = fig.colorbar(im1, ax=ax, shrink=0.82, extend="both")
cbar1.set_label("Average depth to water (m)")

# ---- Average Recharge
ax = axes[1]
im2 = ax.imshow(
    np.ma.masked_invalid(avg_rch),
    origin="upper",
    extent=extent,
    cmap="Blues",
    vmin=rch_vmin,
    vmax=rch_vmax,
)
gdf_bdry.boundary.plot(ax=ax, color="black", linewidth=0.8)
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.text(
    0.02, 0.98, "b)",
    transform=ax.transAxes,
    ha="left", va="top",
    fontsize=14, fontweight="bold"
)
cbar2 = fig.colorbar(im2, ax=ax, shrink=0.82, extend="max")
cbar2.set_label("Average recharge (m/day)")

plt.show()

if save_fig:
    fig.savefig(out_fig, dpi=300, bbox_inches="tight")
    print("Saved:", out_fig)

# ============================================================
# 6) SUMMARY
# ============================================================
print("Average DTW min/max:", np.nanmin(avg_dtw), np.nanmax(avg_dtw))
print("Average recharge min/max:", np.nanmin(avg_rch), np.nanmax(avg_rch))

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import flopy


# =========================================================
# 1) Paths to model outputs
# =========================================================
hds_path = Path(sim_ws) / f"{nameModel}.hds"
cbb_path = Path(sim_ws) / f"{nameModel}.cbb"
lst_path = Path(sim_ws) / f"{nameModel}.lst"

print("HDS exists:", hds_path.exists(), hds_path)
print("CBB exists:", cbb_path.exists(), cbb_path)
print("LST exists:", lst_path.exists(), lst_path)

assert hds_path.exists(), f"Head file not found: {hds_path}"


# =========================================================
# 2) Helper functions
# =========================================================
def get_extent(xorigin, yorigin, delr, delc, nrow, ncol):
    """
    Build plotting extent from model origin + cell sizes.
    """
    if np.ndim(delr) == 0:
        width = ncol * float(delr)
    else:
        width = float(np.sum(delr))

    if np.ndim(delc) == 0:
        height = nrow * float(delc)
    else:
        height = float(np.sum(delc))

    return [xorigin, xorigin + width, yorigin, yorigin + height]


def mask_model_array(arr, idomain_layer=None, huge=1e20):
    """
    Mask MODFLOW placeholder values such as +/-1e30 and inactive cells.
    """
    a = np.array(arr, dtype=float)

    # mask huge positive/negative placeholder values
    a[np.abs(a) >= huge] = np.nan

    # mask inactive cells if provided
    if idomain_layer is not None:
        a = np.where(np.asarray(idomain_layer) > 0, a, np.nan)

    return a


def robust_limits(arr, qlow=2, qhigh=98, symmetric=False):
    a = np.asarray(arr, dtype=float)
    a = a[np.isfinite(a)]
    if a.size == 0:
        return 0.0, 1.0

    vmin = np.nanpercentile(a, qlow)
    vmax = np.nanpercentile(a, qhigh)

    if symmetric:
        vmax_abs = max(abs(vmin), abs(vmax))
        return -vmax_abs, vmax_abs

    return float(vmin), float(vmax)


def get_date_labels(nper, months=None):
    """
    If months exists, label plots as YYYY-MM; otherwise use SP numbers.
    """
    if months is not None and len(months) >= nper:
        return [pd.to_datetime(m).strftime("%Y-%m") for m in months[:nper]]
    return [f"SP {i+1}" for i in range(nper)]


def get_water_table(head3d, idomain3d, huge=1e29):
    """
    Water table = first valid head from the top active layer downward.
    Uses huge=1e29 to catch MODFLOW 6 dry-cell placeholder (1e30).
    """
    nlay, nrow, ncol = head3d.shape
    wt = np.full((nrow, ncol), np.nan, dtype=float)

    for k in range(nlay):
        hk = np.array(head3d[k], dtype=float)

        # mask MODFLOW dry-cell placeholder AND inactive cells
        hk[np.abs(hk) >= huge] = np.nan
        hk[idomain3d[k] <= 0] = np.nan

        # also mask any remaining unrealistic values
        hk[hk > 10000]  = np.nan   # no cell top should exceed 10,000 m
        hk[hk < -1000]  = np.nan   # no head below -1000 m

        take = np.isnan(wt) & np.isfinite(hk)
        wt[take] = hk[take]

    return wt


def extract_head_series(hdobj, kstpkper_list, layer, row, col, idomain3d, huge=1e20):
    """
    Extract a head time series at one model cell.
    """
    vals = []
    for kp in kstpkper_list:
        h = hdobj.get_data(kstpkper=kp)
        v = float(h[layer, row, col])

        if abs(v) >= huge or idomain3d[layer, row, col] <= 0:
            vals.append(np.nan)
        else:
            vals.append(v)

    return np.array(vals, dtype=float)


# =========================================================
# 3) Open head file and inspect saved times
# =========================================================
hdobj = flopy.utils.HeadFile(hds_path)
kstpkper_list = hdobj.get_kstpkper()
times = hdobj.get_times()

print("Number of saved times:", len(times))
print("kstpkper list:", kstpkper_list)

extent = get_extent(xorigin, yorigin, delr, delc, nrow, ncol)
date_labels = get_date_labels(len(kstpkper_list), months=months if "months" in globals() else None)


# =========================================================
# 4) Quick diagnostics: raw vs masked ranges
# =========================================================
print("\n--- HEAD DIAGNOSTICS ---")
for i, kp in enumerate(kstpkper_list):
    h = hdobj.get_data(kstpkper=kp)
    h0_raw = np.array(h[0], dtype=float)
    h0 = mask_model_array(h[0], idomain_layer=idomain[0], huge=1e20)

    print(f"{date_labels[i]}:")
    print("   raw min/max   =", np.nanmin(h0_raw), np.nanmax(h0_raw))
    if np.isfinite(h0).any():
        print("   masked min/max=", np.nanmin(h0), np.nanmax(h0))
        print("   finite count  =", np.isfinite(h0).sum())
    else:
        print("   masked array has no finite values")


# =========================================================
# 5) Summary statistics table for layer 1 heads
# =========================================================
stats = []
for i, kp in enumerate(kstpkper_list):
    h = hdobj.get_data(kstpkper=kp)
    h0 = mask_model_array(h[0], idomain_layer=idomain[0], huge=1e20)

    stats.append({
        "stress_period": i + 1,
        "label": date_labels[i],
        "min_head_L1": np.nanmin(h0),
        "max_head_L1": np.nanmax(h0),
        "mean_head_L1": np.nanmean(h0),
        "std_head_L1": np.nanstd(h0),
    })

df_stats = pd.DataFrame(stats)
print("\n--- LAYER 1 HEAD STATS ---")
print(df_stats)


# =========================================================
# 6) Plot layer 1 heads for selected stress periods
# =========================================================
# choose which stress periods to show
plot_ids = [0, 5, 11]   # first, middle, last for a 12-month run
plot_ids = [i for i in plot_ids if i < len(kstpkper_list)]

head_arrays = []
for idx in plot_ids:
    h = hdobj.get_data(kstpkper=kstpkper_list[idx])
    h0 = mask_model_array(h[0], idomain_layer=idomain[0], huge=1e20)
    head_arrays.append(h0)

vmin, vmax = robust_limits(head_arrays, qlow=2, qhigh=98)
print("\nHead plot vmin/vmax:", vmin, vmax)

fig, axes = plt.subplots(1, len(plot_ids), figsize=(6 * len(plot_ids), 5), squeeze=False)

for ax, idx, arr in zip(axes.flat, plot_ids, head_arrays):
    im = ax.imshow(
        np.ma.masked_invalid(arr),
        origin="upper",
        extent=extent,
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
    )
    ax.set_title(f"Layer 1 head\n{date_labels[idx]}")
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    plt.colorbar(im, ax=ax, shrink=0.8, label="Head (m)")

plt.tight_layout()
plt.show()


# =========================================================
# 7) Plot water table for selected stress periods
# =========================================================
wt_arrays = []
for idx in plot_ids:
    h = hdobj.get_data(kstpkper=kstpkper_list[idx])
    wt = get_water_table(h, idomain, huge=1e20)
    wt_arrays.append(wt)

vmin, vmax = robust_limits(wt_arrays, qlow=2, qhigh=98)
print("Water table plot vmin/vmax:", vmin, vmax)

fig, axes = plt.subplots(1, len(plot_ids), figsize=(6 * len(plot_ids), 5), squeeze=False)

for ax, idx, arr in zip(axes.flat, plot_ids, wt_arrays):
    im = ax.imshow(
        np.ma.masked_invalid(arr),
        origin="upper",
        extent=extent,
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
    )
    ax.set_title(f"Water table\n{date_labels[idx]}")
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    plt.colorbar(im, ax=ax, shrink=0.8, label="Water table (m)")

plt.tight_layout()
plt.show()


# =========================================================
# 8) Plot head change from first to last stress period
# =========================================================
h_first = hdobj.get_data(kstpkper=kstpkper_list[0])
h_last  = hdobj.get_data(kstpkper=kstpkper_list[-1])

h_first_L1 = mask_model_array(h_first[0], idomain_layer=idomain[0], huge=1e20)
h_last_L1  = mask_model_array(h_last[0],  idomain_layer=idomain[0], huge=1e20)

dh = h_last_L1 - h_first_L1

finite_dh = dh[np.isfinite(dh)]
if finite_dh.size > 0:
    lim = np.nanpercentile(np.abs(finite_dh), 98)
else:
    lim = 1.0

plt.figure(figsize=(8, 6))
plt.imshow(
    np.ma.masked_invalid(dh),
    origin="upper",
    extent=extent,
    cmap="RdBu_r",
    vmin=-lim,
    vmax=lim,
)
plt.colorbar(shrink=0.8, label="Head change (m)")
plt.title(f"Layer 1 head change\n{date_labels[0]} to {date_labels[-1]}")
plt.xlabel("X")
plt.ylabel("Y")
plt.tight_layout()
plt.show()

print("\nHead change min/max (m):", np.nanmin(dh), np.nanmax(dh))



# =========================================================
# 10) Save final figures
# =========================================================
fig_dir = Path(r"D:\Users\abolmaal\modelling\Figs\Testing_6")
fig_dir.mkdir(exist_ok=True)

# final water table
wt_last = get_water_table(h_last, idomain, huge=1e20)

plt.figure(figsize=(8, 6))
plt.imshow(
    np.ma.masked_invalid(wt_last),
    origin="upper",
    extent=extent,
    cmap="viridis",
    vmin=np.nanpercentile(wt_last[np.isfinite(wt_last)], 2),
    vmax=np.nanpercentile(wt_last[np.isfinite(wt_last)], 98),
)
plt.colorbar(shrink=0.8, label="Water table (m)")
plt.title(f"Final water table\n{date_labels[-1]}")
plt.xlabel("X")
plt.ylabel("Y")
plt.tight_layout()
plt.savefig(fig_dir / "final_water_table.png", dpi=300, bbox_inches="tight")
plt.show()

# head change
plt.figure(figsize=(8, 6))
plt.imshow(
    np.ma.masked_invalid(dh),
    origin="upper",
    extent=extent,
    cmap="RdBu_r",
    vmin=-lim,
    vmax=lim,
)
plt.colorbar(shrink=0.8, label="Head change (m)")
plt.title(f"Layer 1 head change\n{date_labels[0]} to {date_labels[-1]}")
plt.xlabel("X")
plt.ylabel("Y")
plt.tight_layout()
plt.savefig(fig_dir / "head_change_layer1.png", dpi=300, bbox_inches="tight")
plt.show()

print("\nSaved figures to:", fig_dir)

In [ ]:
import os
import numpy as np
import pandas as pd
import flopy

# ------------------------------------------------------------
# READ HEAD OUTPUT AFTER A SUCCESSFUL RUN
# ------------------------------------------------------------
headfile = os.path.join(sim_ws, f"{nameModel}.hds")

if not os.path.exists(headfile):
    raise FileNotFoundError(
        f"Head file not found: {headfile}\n"
        "Make sure Cell 12 finished with Run success: True before plotting."
    )

hds = flopy.utils.HeadFile(headfile)

# all output times
times = hds.get_times()
print("Number of output times:", len(times))
print("First/last output time:", times[0], times[-1])

# read all heads into one array: shape = (ntime, nlay, nrow, ncol)
head = np.array([hds.get_data(totim=t) for t in times], dtype=float)

# dates matching TDIS monthly periods
dates = pd.date_range(start=START_DATE, periods=head.shape[0], freq="MS")

print("head shape:", head.shape)
print("dates:", dates[0], "to", dates[-1])

# optional plot index for selected maps
max_maps = min(6, head.shape[0])
plot_idx = np.linspace(0, head.shape[0] - 1, max_maps, dtype=int)

# quick sanity check
print("Head min/max:", np.nanmin(np.where(np.abs(head) < 1e20, head, np.nan)),
      np.nanmax(np.where(np.abs(head) < 1e20, head, np.nan)))

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import geopandas as gpd
import flopy
import flopy.utils.binaryfile as bf
from matplotlib.colors import TwoSlopeNorm
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
import matplotlib.font_manager as fm

# ============================================================
# SETTINGS
# ============================================================
SCALEBAR_KM = 100
N_SHOW_MAX  = 6

boundary_shp = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Ibound\extended_Bdry_final_GLB_Albers_exported.shp"

out_fig_ts    = r"D:\Users\abolmaal\modelling\Figs\Testing_6\depthtowatertable.png"
out_fig_maps  = r"D:\Users\abolmaal\modelling\Figs\Testing_6\depthtowater_maps_diverging.png"
out_fig_final = r"D:\Users\abolmaal\modelling\Figs\Testing_6\depthtowater_final_diverging.png"

# Fixed colorbar range in metres
# Negative = water table above land (wetlands), positive = below land
FIXED_RANGE = (-50, 50)

# ============================================================
# FORCE RELOAD — clear all stale variables
# ============================================================
for var in ["head", "dates", "dtw_all", "dtw_final", "dtw_sel"]:
    if var in globals():
        del globals()[var]

head_path = os.path.join(sim_ws, f"{nameModel}.hds")
hds = bf.HeadFile(head_path)
times = hds.get_times()
head = np.array([hds.get_data(totim=t) for t in times], dtype=float)
print("Loaded head array:", head.shape)

# rebuild dates to match head array exactly
dates = pd.date_range(start=START_DATE, periods=head.shape[0], freq="MS")
print("Dates:", dates[0].strftime("%Y-%m"), "to", dates[-1].strftime("%Y-%m"))
print("head periods:", head.shape[0], "| dates:", len(dates))

N_SHOW = min(N_SHOW_MAX, head.shape[0])

# ============================================================
# HELPERS
# ============================================================
top = np.array(gwf.dis.top.array, dtype=float)

def get_depth_to_water(head_t, idomain, top, clip_negative=False):
    wt = get_water_table(head_t, idomain)
    dtw = np.array(top, dtype=float) - wt
    dtw[~np.isfinite(wt)] = np.nan
    if clip_negative:
        dtw = np.where(np.isfinite(dtw), np.maximum(dtw, 0.0), np.nan)
    return dtw

def add_north_arrow(ax, x=0.92, y=0.90, size=0.10):
    ax.annotate("N",
        xy=(x, y), xytext=(x, y - size),
        xycoords="axes fraction", textcoords="axes fraction",
        ha="center", va="center", fontsize=12, fontweight="bold",
        arrowprops=dict(facecolor="black", edgecolor="black",
                        width=3, headwidth=10))

def add_scale_bar(ax, length_km=100, loc="lower left"):
    fontprops = fm.FontProperties(size=10)
    scalebar = AnchoredSizeBar(
        ax.transData, length_km * 1000.0, f"{length_km} km",
        loc=loc, pad=0.4, color="black", frameon=False,
        size_vertical=max(delc[0], delr[0]) * 2.0,
        fontproperties=fontprops)
    ax.add_artist(scalebar)

# ============================================================
# MODEL BOUNDARY
# ============================================================
gdf_bdry = gpd.read_file(boundary_shp)
try:
    if getattr(gwf.modelgrid, "crs", None) is not None:
        gdf_bdry = gdf_bdry.to_crs(gwf.modelgrid.crs)
except Exception:
    pass

plot_idx = np.linspace(0, head.shape[0] - 1, N_SHOW, dtype=int)

# ============================================================
# COLOR NORM — use fixed range
# ============================================================
dtw_vmin, dtw_vmax = FIXED_RANGE
dtw_norm = TwoSlopeNorm(vmin=dtw_vmin, vcenter=0.0, vmax=dtw_vmax)

# ============================================================
# 1) FINAL DEPTH-TO-WATER MAP
# ============================================================
dtw_final = get_depth_to_water(head[-1], idomain, top, clip_negative=False)

fig, ax = plt.subplots(figsize=(9, 7))
pmv = flopy.plot.PlotMapView(model=gwf, ax=ax, layer=0)
pcm = pmv.plot_array(
    np.clip(dtw_final, dtw_vmin, dtw_vmax),
    cmap="RdBu", norm=dtw_norm)

gdf_bdry.boundary.plot(ax=ax, color="black", linewidth=0.8)
ax.grid(False)
ax.set_title(f"Final depth to water\n{dates[-1].strftime('%Y-%m')}")
ax.set_xlabel("X")
ax.set_ylabel("Y")
add_north_arrow(ax)
add_scale_bar(ax, length_km=SCALEBAR_KM, loc="lower left")

cbar = fig.colorbar(pcm, ax=ax, shrink=0.82, extend="both")
cbar.set_label("Depth to water (m)")
plt.tight_layout()
plt.savefig(out_fig_final, dpi=300, bbox_inches="tight")
plt.show()

# ============================================================
# 2) DEPTH-TO-WATER MAPS AT DIFFERENT TIMES
# ============================================================
dtw_sel = [
    get_depth_to_water(head[i], idomain, top, clip_negative=False)
    for i in plot_idx
]

nplots = len(plot_idx)
ncols  = 3
nrows  = int(np.ceil(nplots / ncols))

fig, axes = plt.subplots(nrows, ncols,
                         figsize=(5*ncols, 4.8*nrows),
                         constrained_layout=True)
axes = np.atleast_1d(axes).ravel()
panel_labels = ["a", "b", "c", "d", "e", "f"]

for k, (ax, i, arr) in enumerate(zip(axes, plot_idx, dtw_sel)):
    pmv = flopy.plot.PlotMapView(model=gwf, ax=ax, layer=0)
    pcm = pmv.plot_array(
        np.clip(arr, dtw_vmin, dtw_vmax),
        cmap="RdBu", norm=dtw_norm)

    gdf_bdry.boundary.plot(ax=ax, color="black", linewidth=0.7)
    ax.grid(False)
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_title(f"({panel_labels[k]}) {dates[i].strftime('%Y-%m')}",
                 loc="left", fontweight="bold")
    add_north_arrow(ax, x=0.90, y=0.88, size=0.09)
    add_scale_bar(ax, length_km=SCALEBAR_KM, loc="lower left")

for j in range(len(plot_idx), len(axes)):
    axes[j].axis("off")

cbar = fig.colorbar(pcm, ax=axes.tolist(), shrink=0.82, extend="both")
cbar.set_label("Depth to water (m)")
plt.savefig(out_fig_maps, dpi=300, bbox_inches="tight")
plt.show()

# ============================================================
# 3) BASIN-AVERAGE DEPTH TO WATER THROUGH TIME
# ============================================================
dtw_all = np.array([
    get_depth_to_water(head[i], idomain, top, clip_negative=False)
    for i in range(head.shape[0])
])

# clip extreme artesian values before computing percentiles
# so the time series reflects the meaningful part of the domain
dtw_clipped = np.clip(dtw_all, -50, 500)

mean_dtw = np.nanmean(dtw_clipped, axis=(1, 2))
p05_dtw  = np.nanpercentile(dtw_clipped,  5, axis=(1, 2))
p25_dtw  = np.nanpercentile(dtw_clipped, 25, axis=(1, 2))
p75_dtw  = np.nanpercentile(dtw_clipped, 75, axis=(1, 2))
p95_dtw  = np.nanpercentile(dtw_clipped, 95, axis=(1, 2))

fig, ax = plt.subplots(figsize=(11, 4.8))
ax.fill_between(dates, p25_dtw, p75_dtw,
                alpha=0.25, color="steelblue", label="25–75 percentile")
ax.fill_between(dates, p05_dtw, p95_dtw,
                alpha=0.15, color="steelblue", label="5–95 percentile")
ax.plot(dates, mean_dtw, linewidth=2.0, color="steelblue",
        label="Mean depth to water")
ax.axhline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.5)

ax.set_xlabel("Date")
ax.set_ylabel("Depth to water (m)")
ax.set_title("Basin depth to water through time")
ax.legend()

ax.xaxis.set_major_locator(
    mdates.MonthLocator(interval=2 if len(dates) <= 24 else 12))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(out_fig_ts, dpi=300, bbox_inches="tight")
plt.show()

# ============================================================
# SUMMARY
# ============================================================
print("Saved maps:")
print(out_fig_final)
print(out_fig_maps)
print(out_fig_ts)

active = idomain[0] > 0
print(f"\nFinal DTW stats (active cells, unclipped):")
for pct in [1, 5, 25, 50, 75, 95, 99]:
    print(f"  p{pct:2d}: {np.nanpercentile(dtw_final[active], pct):8.1f} m")
print(f"Percent artesian (DTW < 0): "
      f"{100*np.mean(dtw_final[active] < 0):.1f}%")
print(f"Colorbar limits: {dtw_vmin} to {dtw_vmax} m")